<a href="https://colab.research.google.com/github/PriscilaForcel/DOUTORADO/blob/main/01_BR_UNIVERSO/BR_UNIVERSO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Censo Demográfico 2022, Construção de banco nacional por setor censitário**

**Objetivo:** Desenvolver um procedimento para aquisição, padronização, integração e validação dos resultados do Universo do Censo Demográfico 2022, divulgados pelo Instituto Brasileiro de Geografia e Estatística (IBGE).

**Unidade territorial:** Setor censitário.

**Abrangência:** Brasil, 27 Unidades da Federação.

**Fonte:** IBGE - Censo Demográfico 2022, Agregados por Setores Censitários.

**Conjuntos temáticos:** Básico, Características do Domicílio (Partes 1, 2 e 3), Demografia, Parentesco, Cor ou Raça e Alfabetização.

**Produto esperado:** Banco nacional em formato Parquet, com identificadores territoriais preservados, variáveis identificadas por tema e relatório de validação.

**Aplicação:** Subsidiar pesquisas sobre necessidades habitacionais e desigualdades socioespaciais, incluindo análises voltadas à estimativa e à desagregação territorial do déficit habitacional.

*1. Configuração do ambiente*

Preparar o ambiente para o processamento dos dados do Censo Demográfico 2022. São instaladas e importadas as bibliotecas utilizadas nas etapas de aquisição, tratamento, integração e validação das bases.

Também são identificadas as versões do Python e das principais bibliotecas, além da capacidade de memória RAM e do espaço disponível em disco. Essas informações auxiliam na reprodução dos procedimentos e na definição das estratégias de processamento.

In [ ]:
# 1. CONFIGURAÇÃO DO AMBIENTE

# Instalar as dependências
%pip install -q "duckdb==1.3.2" openpyxl

# Importar as bibliotecas
import os
import sys
import shutil
import zipfile
import urllib.request

import pandas as pd
import duckdb

# Identificar as versões utilizadas
print("CONFIGURAÇÃO DO AMBIENTE")
print("-" * 45)

print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"DuckDB: {duckdb.__version__}")

# Verificar os recursos computacionais
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
disco = shutil.disk_usage("/content")

print(f"\nMemória RAM total: {ram / (1024**3):.2f} GB")
print(f"Espaço livre em disco: {disco.free / (1024**3):.2f} GB")

print("\nAmbiente configurado com sucesso!")


*2. Organização do diretório de trabalho*

Criar diretório temporário no ambiente do Google Colab para armazenar os arquivos utilizados durante o processamento dos dados do Censo Demográfico 2022.

O diretório será utilizado nas etapas de aquisição, extração, padronização e integração das bases. Como o armazenamento local do Colab é temporário, os arquivos necessários à preservação dos resultados serão salvos, posteriormente, no Google Drive.

A separação entre armazenamento temporário e persistente permite organizar o processamento e reduzir operações de leitura e escrita diretamente no Google Drive.

In [ ]:
# 2. ORGANIZAÇÃO DO DIRETÓRIO DE TRABALHO

# Definir o diretório temporário do projeto
pasta_local = "/content/censo2022"

# Criar o diretório, caso ainda não exista
os.makedirs(pasta_local, exist_ok=True)

# Conferir sua localização
print("DIRETÓRIO DE TRABALHO")
print("-" * 45)

print("Localização:", pasta_local)
print("Diretório disponível:", os.path.isdir(pasta_local))

# Listar arquivos existentes, sem excluir nenhum
arquivos = os.listdir(pasta_local)

print("Arquivos encontrados:", len(arquivos))

for arquivo in sorted(arquivos):
    print(" -", arquivo)


*3. Conexão com o Google Drive*

Conectar o Google Drive ao ambiente do Google Colab para permitir o armazenamento persistente dos arquivos originais, das bases processadas e dos resultados das validações.

Diferentemente do armazenamento temporário em /content/, os arquivos salvos no Google Drive permanecem disponíveis após o encerramento da sessão.

A conexão exige autorização do usuário e não concede acesso público aos arquivos armazenados.

In [ ]:
# 3. CONEXÃO COM O GOOGLE DRIVE

from google.colab import drive

# Conectar o Google Drive ao ambiente Colab
drive.mount("/content/drive")

# Confirmar a conexão
print("Google Drive conectado com sucesso!")


*4. Organização dos diretórios no Google Drive*

Definir estrutura padronizada de diretórios para o armazenamento persistente dos dados do Censo Demográfico 2022 no Google Drive do usuário.

Os diretórios são organizados para separar os arquivos originais disponibilizados pelo IBGE, os dados processados e os resultados das validações.

Os arquivos originais serão obtidos diretamente das fontes oficiais do IBGE e armazenados para reutilização em execuções posteriores, evitando downloads e processamentos desnecessários.

Essa organização permite a reprodução do procedimento por outros pesquisadores, sem depender de arquivos ou diretórios pessoais.

In [ ]:
# 4. ORGANIZAÇÃO DOS DIRETÓRIOS NO GOOGLE DRIVE


# Diretório principal do projeto
# Cada pesquisador pode adaptar este caminho ao seu Google Drive.

pasta_drive = (
    "/content/drive/MyDrive/DOUTORADO/IBGE_UNIVERSO"
)


# Diretórios de armazenamento
pasta_originais = os.path.join(
    pasta_drive, "DADOS_ORIGINAIS_2022"
)

pasta_tratados = os.path.join(
    pasta_drive, "DADOS_TRATADOS_2022"
)

pasta_resultados = os.path.join(
    pasta_drive, "RESULTADOS_2022"
)

# Criar os diretórios, caso não existam
diretorios = {
    "Projeto": pasta_drive,
    "Arquivos originais": pasta_originais,
    "Dados tratados": pasta_tratados,
    "Resultados": pasta_resultados
}

for nome, caminho in diretorios.items():
    os.makedirs(caminho, exist_ok=True)

# Verificar os diretórios
print("DIRETÓRIOS CONFIGURADOS")
print("-" * 50)

for nome, caminho in diretorios.items():
    status = "OK" if os.path.isdir(caminho) else "ERRO"
    print(f"{nome}: {status}")
    print(f"  {caminho}\n")


*5. Aquisição dos arquivos originais do IBGE*

Realizar a aquisição dos arquivos nacionais dos Agregados por Setores Censitários do Censo Demográfico 2022, disponibilizados pelo Instituto Brasileiro de Geografia e Estatística (IBGE).

*São selecionados oito conjuntos:* Básico, Características do Domicílio (Partes 1, 2 e 3), Demografia, Parentesco, Cor ou Raça e Alfabetização. Os conjuntos específicos de populações indígenas e quilombolas e os dados de óbitos não integram esta versão da pesquisa.

O procedimento verifica a existência e a integridade dos arquivos compactados antes de realizar novos downloads. Os arquivos originais são preservados, permitindo sua reutilização e a rastreabilidade das fontes.

*Fonte:* IBGE, Censo Demográfico 2022, Agregados por Setores Censitários, arquivos nacionais em formato CSV compactado.

In [ ]:
# 5. AQUISIÇÃO DOS ARQUIVOS ORIGINAIS DO IBGE

# Endereço oficial dos arquivos CSV compactados
url_base = (
    "https://ftp.ibge.gov.br/Censos/"
    "Censo_Demografico_2022/"
    "Agregados_por_Setores_Censitarios/"
    "Agregados_por_Setor_csv/"
)

# Conjuntos selecionados para a pesquisa
arquivos_ibge = {
    "BASICO": "Agregados_por_setores_basico_BR_20260520.zip",
    "DOM1": "Agregados_por_setores_caracteristicas_domicilio1_BR_20261021.zip",
    "DOM2": "Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.zip",
    "DOM3": "Agregados_por_setores_caracteristicas_domicilio3_BR_20250417.zip",
    "DEM": "Agregados_por_setores_demografia_BR.zip",
    "PAR": "Agregados_por_setores_parentesco_BR.zip",
    "COR": "Agregados_por_setores_cor_ou_raca_BR.zip",
    "ALF": "Agregados_por_setores_alfabetizacao_BR.zip"
}

# Verificar a integridade de um arquivo ZIP
def zip_valido(caminho):
    if not os.path.isfile(caminho):
        return False

    try:
        with zipfile.ZipFile(caminho, "r") as arquivo:
            return arquivo.testzip() is None
    except (OSError, zipfile.BadZipFile):
        return False


print("AQUISIÇÃO DOS DADOS DO IBGE")
print("-" * 55)

for tema, nome_arquivo in arquivos_ibge.items():

    destino = os.path.join(pasta_originais, nome_arquivo)

    # Reutilizar arquivos existentes e válidos
    if zip_valido(destino):
        print(f"{tema}: arquivo existente e válido.")
        continue

    # Baixar o arquivo para o armazenamento temporário
    print(f"{tema}: iniciando download...")

    temporario = os.path.join(
        pasta_local, nome_arquivo + ".parcial"
    )

    destino_parcial = destino + ".parcial"

    try:
        urllib.request.urlretrieve(
            url_base + nome_arquivo,
            temporario
        )

        # Validar o arquivo baixado
        if not zip_valido(temporario):
            raise ValueError("O arquivo ZIP está corrompido.")

        # Salvar no Google Drive
        shutil.copy2(temporario, destino_parcial)
        os.replace(destino_parcial, destino)

        tamanho_mb = os.path.getsize(destino) / (1024**2)

        print(f"  Concluído: {tamanho_mb:.2f} MB")

    except Exception as erro:
        print(f"  ERRO ao obter {tema}: {erro}")
        raise

    finally:
        # Remover somente arquivos temporários
        for caminho in [temporario, destino_parcial]:
            if os.path.exists(caminho):
                os.remove(caminho)

print("\nAquisição concluída!")


*6. Inspeção dos arquivos compactados*

Examinar a estrutura interna dos arquivos ZIP obtidos do IBGE, identificando os arquivos CSV disponíveis, seus nomes e tamanhos descompactados.

A inspeção permite verificar a organização dos conjuntos temáticos antes da leitura e conversão para Parquet, sem extrair os arquivos ou modificar seu conteúdo.

Os resultados serão utilizados para definir um procedimento padronizado de leitura dos dados.

In [ ]:
# 6. INSPEÇÃO DOS ARQUIVOS COMPACTADOS

registros_zip = []

for tema, nome_zip in arquivos_ibge.items():

    caminho_zip = os.path.join(
        pasta_originais, nome_zip
    )

    print(f"\n{tema} | {nome_zip}")
    print("-" * 65)

    with zipfile.ZipFile(caminho_zip, "r") as arquivo:

        for item in arquivo.infolist():

            if item.is_dir():
                continue

            tamanho_mb = item.file_size / (1024**2)

            registros_zip.append({
                "Tema": tema,
                "Arquivo interno": item.filename,
                "Tamanho (MB)": round(tamanho_mb, 2)
            })

            print(f"Arquivo: {item.filename}")
            print(f"Tamanho: {tamanho_mb:.2f} MB")

# Organizar o inventário
df_inventario = pd.DataFrame(registros_zip)

print("\nRESUMO DO INVENTÁRIO")
print("-" * 65)

print(f"Conjuntos temáticos: {len(arquivos_ibge)}")
print(f"Arquivos internos: {len(df_inventario)}")

display(df_inventario)


*7. Identificação da estrutura dos arquivos CSV*

Examinar a estrutura dos oito arquivos CSV nacionais, considerando a codificação dos caracteres, o delimitador, os nomes das colunas e a organização dos identificadores territoriais.

A leitura é realizada diretamente dos arquivos ZIP, utilizando apenas o cabeçalho e uma pequena amostra dos registros, sem carregar integralmente as bases na memória.

Esse diagnóstico orienta a padronização dos arquivos e a posterior conversão para Parquet, preservando os identificadores territoriais e os valores originais das variáveis.

In [ ]:
# 7. IDENTIFICAÇÃO DA ESTRUTURA DOS ARQUIVOS CSV

import csv
import io

diagnostico_csv = []

for tema, nome_zip in arquivos_ibge.items():

    caminho_zip = os.path.join(
        pasta_originais, nome_zip
    )

    with zipfile.ZipFile(caminho_zip, "r") as arquivo:

        # Selecionar o CSV dentro do ZIP
        arquivos_csv = [
            nome for nome in arquivo.namelist()
            if nome.lower().endswith(".csv")
        ]

        if len(arquivos_csv) != 1:
            raise ValueError(
                f"{tema}: esperado 1 CSV, encontrados "
                f"{len(arquivos_csv)}."
            )

        nome_csv = arquivos_csv[0]

        # Ler uma pequena amostra de bytes
        with arquivo.open(nome_csv) as dados:
            amostra = dados.read(65536)

        # Identificar uma codificação compatível
        codificacao = None

        for tentativa in ["utf-8-sig", "cp1252", "latin-1"]:
            try:
                texto = amostra.decode(tentativa)
                codificacao = tentativa
                break
            except UnicodeDecodeError:
                continue

        # Identificar o delimitador no cabeçalho
        primeira_linha = texto.splitlines()[0]

        delimitador = max(
            [";", ",", "\t", "|"],
            key=lambda sep: primeira_linha.count(sep)
        )

        # Ler cabeçalho e duas linhas de exemplo
        with arquivo.open(nome_csv) as dados:
            with io.TextIOWrapper(
                dados, encoding=codificacao, newline=""
            ) as texto_csv:

                leitor = csv.reader(
                    texto_csv, delimiter=delimitador
                )

                colunas = next(leitor)
                exemplos = []

                for _ in range(2):
                    linha = next(leitor, None)
                    if linha is None:
                        break
                    exemplos.append(linha[:5])

        diagnostico_csv.append({
            "Tema": tema,
            "Codificação": codificacao,
            "Delimitador": repr(delimitador),
            "Colunas": len(colunas),
            "Primeira coluna": colunas[0],
            "Exemplo": exemplos
        })

# Apresentar resultados
df_diagnostico = pd.DataFrame(diagnostico_csv)

print("DIAGNÓSTICO DOS ARQUIVOS CSV")
print("-" * 60)

display(
    df_diagnostico.drop(columns=["Exemplo"])
)

print("\nEXEMPLOS DOS IDENTIFICADORES")
print("-" * 60)

for registro in diagnostico_csv:
    print(f"\n{registro['Tema']}")
    print("Primeiras colunas e valores:", registro["Exemplo"])


*8. Padronização dos nomes das variáveis*

Estabelecer os critérios de identificação das variáveis que comporão o banco nacional integrado.

O identificador territorial será padronizado como `CD_SETOR` em todos os conjuntos de dados, preservando os códigos originais dos setores censitários como texto.

Para evitar conflitos entre variáveis de diferentes conjuntos, os nomes das colunas temáticas receberão prefixos correspondentes às suas fontes (`DOM1`, `DOM2`, `DOM3`, `DEM`, `PAR`, `COR` e `ALF`). As variáveis do arquivo Básico permanecerão com suas denominações originais.

A padronização dos nomes será realizada sem modificar os valores das variáveis. Nesta etapa, serão verificados apenas os cabeçalhos dos arquivos, sem carregar integralmente as bases na memória.

In [ ]:
# 8. PADRONIZAÇÃO DOS NOMES DAS VARIÁVEIS

import csv
import io

# Configuração dos identificadores originais
chaves_originais = {
    "BASICO": "CD_SETOR",
    "DOM1": "setor",
    "DOM2": "setor",
    "DOM3": "setor",
    "DEM": "CD_setor",
    "PAR": "CD_SETOR",
    "COR": "CD_SETOR",
    "ALF": "CD_setor"
}

# Codificações identificadas na Célula 7
codificacoes = {
    "BASICO": "cp1252",
    "DOM1": "utf-8-sig",
    "DOM2": "utf-8-sig",
    "DOM3": "utf-8-sig",
    "DEM": "utf-8-sig",
    "PAR": "utf-8-sig",
    "COR": "utf-8-sig",
    "ALF": "utf-8-sig"
}

# Armazenar a correspondência dos nomes das colunas
esquemas = {}
registros_colunas = []

for tema, nome_zip in arquivos_ibge.items():

    caminho_zip = os.path.join(
        pasta_originais, nome_zip
    )

    with zipfile.ZipFile(caminho_zip, "r") as arquivo:

        nomes_csv = [
            nome for nome in arquivo.namelist()
            if nome.lower().endswith(".csv")
        ]

        if len(nomes_csv) != 1:
            raise ValueError(f"{tema}: CSV não identificado.")

        with arquivo.open(nomes_csv[0]) as dados:
            with io.TextIOWrapper(
                dados,
                encoding=codificacoes[tema],
                newline=""
            ) as texto:
                colunas = next(csv.reader(texto, delimiter=";"))

    chave = chaves_originais[tema]

    if colunas.count(chave) != 1:
        raise ValueError(
            f"{tema}: identificador {chave} ausente ou duplicado."
        )

    if len(colunas) != len(set(colunas)):
        raise ValueError(
            f"{tema}: existem nomes de colunas duplicados."
        )

    renomear = {}

    for coluna in colunas:

        if coluna == chave:
            novo_nome = "CD_SETOR"

        elif tema == "BASICO":
            novo_nome = coluna

        else:
            novo_nome = f"{tema}_{coluna}"

        renomear[coluna] = novo_nome

        registros_colunas.append({
            "Tema": tema,
            "Nome original": coluna,
            "Nome padronizado": novo_nome
        })

    esquemas[tema] = renomear

# Construir catálogo preliminar das colunas
df_colunas = pd.DataFrame(registros_colunas)

# Verificar conflitos após padronização
nomes_finais = df_colunas[
    df_colunas["Nome padronizado"] != "CD_SETOR"
]["Nome padronizado"]

if nomes_finais.duplicated().any():
    raise ValueError(
        "Existem conflitos entre os nomes padronizados."
    )

# Conferir quantidade de colunas da integração
total_final = len(nomes_finais) + 1

print("PADRONIZAÇÃO DOS NOMES DAS VARIÁVEIS")
print("-" * 55)

print("Identificador único: CD_SETOR")
print("Colunas previstas:", total_final)
print("Conflitos entre variáveis:", nomes_finais.duplicated().sum())

assert total_final == 1356, (
    f"Esperadas 1.356 colunas, encontradas {total_final}."
)

print("\nExemplos da padronização:")
display(df_colunas.groupby("Tema").head(3))

print("\nEstrutura de colunas validada!")


*9. Validação dos identificadores territoriais*

Verificar a integridade dos códigos dos setores censitários nos oito conjuntos de dados do Censo Demográfico 2022.

São analisados o número total de registros, a quantidade de setores distintos, a presença de códigos ausentes, a existência de duplicidades e a conformidade dos identificadores com a estrutura de 15 dígitos.

O arquivo Básico constitui a referência territorial do projeto. Assim, todos os seus setores deverão ser preservados na integração final, independentemente da disponibilidade de informações nos demais conjuntos temáticos.

A leitura é realizada em blocos para reduzir o consumo de memória, mantendo os identificadores como texto e sem modificar os arquivos originais.

In [ ]:
# 9. VALIDAÇÃO DOS IDENTIFICADORES TERRITORIAIS

import io

resultados_validacao = []
setores_por_tema = {}

for tema, nome_zip in arquivos_ibge.items():

    caminho_zip = os.path.join(
        pasta_originais, nome_zip
    )

    chave = chaves_originais[tema]
    codificacao = codificacoes[tema]

    total_registros = 0
    codigos_ausentes = 0
    codigos_invalidos = 0
    setores = set()

    print(f"Verificando {tema}...")

    with zipfile.ZipFile(caminho_zip, "r") as arquivo:

        nome_csv = [
            nome for nome in arquivo.namelist()
            if nome.lower().endswith(".csv")
        ][0]

        with arquivo.open(nome_csv) as dados:

            with io.TextIOWrapper(
                dados,
                encoding=codificacao,
                newline=""
            ) as texto:

                blocos = pd.read_csv(
                    texto,
                    sep=";",
                    usecols=[chave],
                    dtype=str,
                    chunksize=50000,
                    keep_default_na=False
                )

                for bloco in blocos:

                    codigos = bloco[chave]

                    total_registros += len(codigos)

                    ausentes = codigos.eq("")
                    validos = codigos.str.fullmatch(
                        r"\d{15}"
                    )

                    codigos_ausentes += int(
                        ausentes.sum()
                    )

                    codigos_invalidos += int(
                        (~validos & ~ausentes).sum()
                    )

                    setores.update(
                        codigos[validos].tolist()
                    )

    distintos = len(setores)

    duplicidades = (
        total_registros
        - codigos_ausentes
        - codigos_invalidos
        - distintos
    )

    setores_por_tema[tema] = setores

    resultados_validacao.append({
        "Tema": tema,
        "Registros": total_registros,
        "Setores distintos": distintos,
        "Códigos ausentes": codigos_ausentes,
        "Códigos inválidos": codigos_invalidos,
        "Duplicidades": duplicidades
    })

# Organizar os resultados
df_validacao_setores = pd.DataFrame(
    resultados_validacao
)

print("\nVALIDAÇÃO DOS SETORES CENSITÁRIOS")
print("-" * 65)

display(df_validacao_setores)

# Verificar a integridade das chaves
assert (
    df_validacao_setores[
        ["Códigos ausentes", "Códigos inválidos", "Duplicidades"]
    ].to_numpy() == 0
).all(), "Foram identificados problemas nos identificadores."

print("\nIdentificadores validados!")


*10. Verificação da cobertura territorial dos conjuntos de dados*

Comparar os identificadores territoriais dos sete conjuntos temáticos com aqueles presentes no arquivo Básico do Censo Demográfico 2022.

A análise verifica se todos os setores dos conjuntos temáticos estão contemplados no Básico, identifica os setores sem correspondência e avalia a consistência da cobertura territorial entre os diferentes conjuntos.

O arquivo Básico constitui a referência territorial da integração. Assim, os setores presentes exclusivamente nesse conjunto serão preservados na base final, com valores ausentes para as variáveis temáticas não disponibilizadas.

Essa verificação é necessária para assegurar que a integração mantenha a cobertura territorial prevista, sem exclusão ou duplicação de setores censitários.

In [ ]:
# 10. VERIFICAÇÃO DA COBERTURA TERRITORIAL

# Conjunto de setores do arquivo Básico
setores_basico = setores_por_tema["BASICO"]

# Conjuntos temáticos selecionados
temas = [
    "DOM1", "DOM2", "DOM3",
    "DEM", "PAR", "COR", "ALF"
]

resultados_cobertura = []

for tema in temas:

    setores_tema = setores_por_tema[tema]

    # Setores do Básico sem correspondência temática
    somente_basico = setores_basico - setores_tema

    # Setores temáticos ausentes do Básico
    somente_tema = setores_tema - setores_basico

    # Setores presentes nos dois conjuntos
    correspondentes = setores_basico & setores_tema

    resultados_cobertura.append({
        "Tema": tema,
        "Setores correspondentes": len(correspondentes),
        "Apenas no Básico": len(somente_basico),
        "Apenas no temático": len(somente_tema)
    })

# Organizar resultados
df_cobertura = pd.DataFrame(resultados_cobertura)

print("COBERTURA TERRITORIAL DOS CONJUNTOS")
print("-" * 65)

display(df_cobertura)

# Verificar se todos os temas possuem a mesma cobertura
referencia = setores_por_tema["DOM1"]

mesma_cobertura = all(
    setores_por_tema[tema] == referencia
    for tema in temas
)

print("\nVERIFICAÇÕES")
print("-" * 65)

print("Cobertura temática idêntica:", mesma_cobertura)

print(
    "Todos os setores temáticos estão no Básico:",
    (df_cobertura["Apenas no temático"] == 0).all()
)

# Interromper caso haja inconsistências
assert mesma_cobertura, (
    "Os conjuntos temáticos apresentam coberturas diferentes."
)

assert (
    df_cobertura["Apenas no temático"] == 0
).all(), (
    "Existem setores temáticos ausentes do Básico."
)

print("\nCobertura territorial validada!")


*11. Conversão do arquivo Básico para Parquet*

Converter o arquivo Básico do Censo Demográfico 2022, disponibilizado em CSV compactado, para o formato Parquet.

O arquivo é lido diretamente do ZIP, em blocos, para limitar o consumo de memória RAM. Os identificadores territoriais e os valores das variáveis são preservados como texto, sem transformações numéricas nesta etapa.

O formato Parquet será utilizado nas etapas posteriores de integração e análise, permitindo maior eficiência na leitura de colunas e no processamento dos dados.

Após a conversão, são verificados o número de registros, a quantidade de colunas e a integridade do identificador `CD_SETOR`. O arquivo original do IBGE permanece inalterado.

In [ ]:
# 11. CONVERSÃO DO ARQUIVO BÁSICO PARA PARQUET

# Biblioteca para escrita de arquivos Parquet
%pip install -q pyarrow

import pyarrow as pa
import pyarrow.parquet as pq

# Definir o conjunto a processar
tema = "BASICO"

# Localizar o ZIP original
caminho_zip = os.path.join(
    pasta_originais,
    arquivos_ibge[tema]
)

# Definir os arquivos de saída
nome_saida = "BASICO_2022.parquet"

saida_local = os.path.join(
    pasta_local,
    nome_saida
)

saida_drive = os.path.join(
    pasta_tratados,
    nome_saida
)

# Preservar a ordem original das colunas
nomes_originais = list(esquemas[tema].keys())
nomes_padronizados = list(esquemas[tema].values())

# Esquema de saída: todas as colunas como texto
schema = pa.schema([
    pa.field(nome, pa.string())
    for nome in nomes_padronizados
])

total_registros = 0
escritor = None

print("CONVERSÃO DO ARQUIVO BÁSICO")
print("-" * 55)

try:

    with zipfile.ZipFile(caminho_zip, "r") as arquivo:

        nomes_csv = [
            nome for nome in arquivo.namelist()
            if nome.lower().endswith(".csv")
        ]

        if len(nomes_csv) != 1:
            raise ValueError(
                "O ZIP deve conter exatamente um CSV."
            )

        with arquivo.open(nomes_csv[0]) as dados:

            with io.TextIOWrapper(
                dados,
                encoding=codificacoes[tema],
                newline=""
            ) as texto:

                blocos = pd.read_csv(
                    texto,
                    sep=";",
                    dtype=str,
                    chunksize=10000,
                    keep_default_na=False,
                    na_filter=False
                )

                escritor = pq.ParquetWriter(
                    saida_local,
                    schema=schema,
                    compression="zstd"
                )

                for bloco in blocos:

                    # Conferir a estrutura original
                    if list(bloco.columns) != nomes_originais:
                        raise ValueError(
                            "As colunas do CSV não correspondem "
                            "ao esquema validado."
                        )

                    # Aplicar nomes padronizados
                    bloco = bloco.rename(
                        columns=esquemas[tema]
                    )

                    # Converter o bloco para Arrow
                    tabela = pa.Table.from_pandas(
                        bloco,
                        schema=schema,
                        preserve_index=False
                    )

                    # Escrever no Parquet
                    escritor.write_table(tabela)

                    total_registros += len(bloco)

finally:
    if escritor is not None:
        escritor.close()

# Conferir o arquivo gerado
metadados = pq.ParquetFile(saida_local)

linhas = metadados.metadata.num_rows
colunas = len(metadados.schema_arrow.names)

# Validar a estrutura
assert linhas == 468099, (
    f"Quantidade inesperada de registros: {linhas}"
)

assert colunas == 38, (
    f"Quantidade inesperada de colunas: {colunas}"
)

assert total_registros == linhas

# Validar a unicidade da chave
con_validacao = duckdb.connect(":memory:")

try:
    consulta = con_validacao.execute(
        """
        SELECT
            COUNT(*) AS total,
            COUNT(DISTINCT CD_SETOR) AS distintos
        FROM read_parquet(?)
        """,
        [saida_local]
    ).fetchone()
finally:
    con_validacao.close()

assert consulta == (468099, 468099), (
    "Os identificadores não são únicos."
)

# Salvar no Google Drive
saida_parcial = saida_drive + ".parcial"

shutil.copy2(saida_local, saida_parcial)

# Verificar a cópia antes de torná-la definitiva
copia = pq.ParquetFile(saida_parcial)

assert copia.metadata.num_rows == linhas
assert len(copia.schema_arrow.names) == colunas

os.replace(saida_parcial, saida_drive)

print("\nARQUIVO PARQUET VALIDADO")
print("-" * 55)
print("Registros:", linhas)
print("Colunas:", colunas)
print("Setores distintos:", consulta[1])

print(
    "Tamanho:",
    f"{os.path.getsize(saida_drive) / (1024**2):.2f} MB"
)

print("Arquivo salvo em:", saida_drive)
print("\nConversão concluída com sucesso!")


*12. Conversão dos conjuntos temáticos para Parquet*

Converter os sete conjuntos temáticos selecionados do Censo Demográfico 2022 para o formato Parquet: Características do Domicílio (Partes 1, 2 e 3), Demografia, Parentesco, Cor ou Raça e Alfabetização.

Os arquivos CSV são lidos diretamente de seus respectivos ZIPs, em blocos de 10.000 registros, sem carregar integralmente as bases na memória.

Durante a conversão, os identificadores territoriais são padronizados como `CD_SETOR`, e as variáveis recebem prefixos correspondentes aos conjuntos de origem. Os valores originais são preservados como texto, inclusive aqueles protegidos por sigilo estatístico (`X`).

Cada arquivo convertido é validado quanto ao número de registros, à quantidade de colunas e à unicidade dos identificadores. Os resultados são armazenados separadamente para posterior integração nacional.

Os arquivos originais do IBGE permanecem inalterados.

In [ ]:
# 12. CONVERSÃO DOS CONJUNTOS TEMÁTICOS PARA PARQUET

temas_tematicos = [
    "DOM1", "DOM2", "DOM3",
    "DEM", "PAR", "COR", "ALF"
]

resultados_conversao = []

for tema in temas_tematicos:

    print(f"\nPROCESSANDO {tema}")
    print("-" * 55)

    # Localizar o arquivo original e definir as saídas
    caminho_zip = os.path.join(
        pasta_originais, arquivos_ibge[tema]
    )

    nome_saida = f"{tema}_2022.parquet"

    saida_local = os.path.join(
        pasta_local, nome_saida
    )

    saida_drive = os.path.join(
        pasta_tratados, nome_saida
    )

    saida_parcial = saida_drive + ".parcial"

    # Definir a estrutura padronizada
    nomes_originais = list(esquemas[tema].keys())
    nomes_padronizados = list(esquemas[tema].values())

    schema = pa.schema([
        pa.field(nome, pa.string())
        for nome in nomes_padronizados
    ])

    # Quantidades esperadas da validação anterior
    esperado_linhas = len(setores_por_tema[tema])
    esperado_colunas = len(nomes_padronizados)

    escritor = None
    total_lido = 0

    try:
        # Ler diretamente o CSV contido no ZIP
        with zipfile.ZipFile(caminho_zip, "r") as arquivo:

            nomes_csv = [
                nome for nome in arquivo.namelist()
                if nome.lower().endswith(".csv")
            ]

            if len(nomes_csv) != 1:
                raise ValueError(
                    f"{tema}: esperado exatamente um CSV."
                )

            with arquivo.open(nomes_csv[0]) as dados:

                with io.TextIOWrapper(
                    dados,
                    encoding=codificacoes[tema],
                    newline=""
                ) as texto:

                    blocos = pd.read_csv(
                        texto,
                        sep=";",
                        dtype=str,
                        chunksize=10000,
                        keep_default_na=False,
                        na_filter=False
                    )

                    escritor = pq.ParquetWriter(
                        saida_local,
                        schema=schema,
                        compression="zstd"
                    )

                    for bloco in blocos:

                        # Validar o cabeçalho original
                        if list(bloco.columns) != nomes_originais:
                            raise ValueError(
                                f"{tema}: colunas inesperadas."
                            )

                        # Padronizar os nomes
                        bloco = bloco.rename(
                            columns=esquemas[tema]
                        )

                        # Escrever o bloco no Parquet
                        tabela = pa.Table.from_pandas(
                            bloco,
                            schema=schema,
                            preserve_index=False
                        )

                        escritor.write_table(tabela)

                        total_lido += len(bloco)

        # Fechar o arquivo antes de validá-lo
        escritor.close()
        escritor = None

        # Validar registros e colunas
        metadados = pq.ParquetFile(saida_local)

        linhas = metadados.metadata.num_rows
        colunas = len(metadados.schema_arrow.names)

        if linhas != esperado_linhas:
            raise ValueError(
                f"{tema}: esperado {esperado_linhas} "
                f"registros, encontrados {linhas}."
            )

        if colunas != esperado_colunas:
            raise ValueError(
                f"{tema}: esperado {esperado_colunas} "
                f"colunas, encontradas {colunas}."
            )

        if total_lido != linhas:
            raise ValueError(
                f"{tema}: divergência na leitura dos registros."
            )

        # Validar os identificadores
        con_validacao = duckdb.connect(":memory:")

        try:
            total, distintos, invalidos = con_validacao.execute(
                """
                SELECT
                    COUNT(*),
                    COUNT(DISTINCT CD_SETOR),
                    COUNT(*) FILTER (
                        WHERE NOT regexp_full_match(
                            CD_SETOR, '[0-9]{15}'
                        )
                        OR CD_SETOR IS NULL
                    )
                FROM read_parquet(?)
                """,
                [saida_local]
            ).fetchone()
        finally:
            con_validacao.close()

        if (
            total != esperado_linhas
            or distintos != esperado_linhas
            or invalidos != 0
        ):
            raise ValueError(
                f"{tema}: identificadores inconsistentes."
            )

        # Copiar para o Drive e conferir a cópia
        shutil.copy2(saida_local, saida_parcial)

        copia = pq.ParquetFile(saida_parcial)

        if (
            copia.metadata.num_rows != linhas
            or len(copia.schema_arrow.names) != colunas
        ):
            raise ValueError(
                f"{tema}: cópia do Parquet inconsistente."
            )

        os.replace(saida_parcial, saida_drive)

        tamanho_mb = (
            os.path.getsize(saida_drive) / (1024**2)
        )

        resultados_conversao.append({
            "Tema": tema,
            "Registros": linhas,
            "Colunas": colunas,
            "Setores distintos": distintos,
            "Tamanho (MB)": round(tamanho_mb, 2)
        })

        print(f"Registros: {linhas:,}")
        print(f"Colunas: {colunas}")
        print(f"Setores distintos: {distintos:,}")
        print(f"Tamanho: {tamanho_mb:.2f} MB")
        print("Status: VALIDADO")

    finally:
        if escritor is not None:
            escritor.close()

        # Excluir somente os arquivos temporários
        for caminho in [saida_local, saida_parcial]:
            if os.path.isfile(caminho):
                os.remove(caminho)

# Resumo dos resultados
df_conversao = pd.DataFrame(resultados_conversao)

print("\nRESUMO DA CONVERSÃO DOS CONJUNTOS TEMÁTICOS")
print("-" * 65)

display(df_conversao)

print("\nConversões concluídas e validadas!")


*13. Inventário e validação dos arquivos Parquet*

Verificar a disponibilidade e a estrutura dos oito arquivos Parquet produzidos a partir dos dados originais do Censo Demográfico 2022.

São conferidos o número de registros, a quantidade de colunas, a presença do identificador territorial `CD_SETOR` e a correspondência entre os nomes das variáveis e o esquema previamente definido.

A validação utiliza os metadados dos arquivos Parquet, sem carregar integralmente os conjuntos na memória. Essa verificação constitui uma etapa preparatória para a integração nacional por setor censitário.

In [ ]:
# 13. INVENTÁRIO E VALIDAÇÃO DOS ARQUIVOS PARQUET

# Conjuntos selecionados, na ordem da integração
temas_banco = [
    "BASICO", "DOM1", "DOM2", "DOM3",
    "DEM", "PAR", "COR", "ALF"
]

# Armazenar os caminhos para as próximas etapas
caminhos_parquet = {}
registros_inventario = []

for tema in temas_banco:

    caminho = os.path.join(
        pasta_tratados,
        f"{tema}_2022.parquet"
    )

    if not os.path.isfile(caminho):
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}"
        )

    # Consultar apenas os metadados
    parquet = pq.ParquetFile(caminho)

    linhas = parquet.metadata.num_rows
    colunas = parquet.schema_arrow.names

    # Estrutura definida na Célula 8
    colunas_esperadas = list(
        esquemas[tema].values()
    )

    linhas_esperadas = len(
        setores_por_tema[tema]
    )

    # Verificar a estrutura
    if linhas != linhas_esperadas:
        raise ValueError(
            f"{tema}: número de registros divergente."
        )

    if colunas != colunas_esperadas:
        raise ValueError(
            f"{tema}: esquema de colunas divergente."
        )

    if colunas.count("CD_SETOR") != 1:
        raise ValueError(
            f"{tema}: identificador territorial inválido."
        )

    caminhos_parquet[tema] = caminho

    registros_inventario.append({
        "Tema": tema,
        "Registros": linhas,
        "Colunas": len(colunas),
        "Tamanho (MB)": round(
            os.path.getsize(caminho) / (1024**2), 2
        ),
        "Status": "OK"
    })

# Organizar o inventário
df_parquets = pd.DataFrame(
    registros_inventario
)

print("INVENTÁRIO DOS ARQUIVOS PARQUET")
print("-" * 65)

display(df_parquets)

# Calcular o número de colunas do banco integrado
total_colunas = (
    df_parquets["Colunas"].sum()
    - (len(temas_banco) - 1)
)

print("\nESTRUTURA PREVISTA DO BANCO NACIONAL")
print("-" * 65)

print(
    "Setores censitários:",
    len(setores_por_tema["BASICO"])
)

print("Colunas previstas:", total_colunas)

assert total_colunas == 1356

print("\nArquivos disponíveis e estruturas validadas!")


*14. Preparação da integração nacional por setor censitário*

Preparar os arquivos Parquet para a integração nacional dos dados do Censo Demográfico 2022.

Os oito conjuntos são copiados para o armazenamento temporário do Google Colab, a fim de reduzir as operações de leitura diretamente no Google Drive.

O arquivo Básico é utilizado para definir lotes de processamento, compostos inicialmente por até 10.000 setores censitários, ordenados pelo identificador `CD_SETOR`.

A divisão em lotes constitui uma estratégia computacional para limitar o uso da memória RAM. Ela não altera a unidade territorial de análise nem implica agregação ou exclusão de registros.

Cada lote será posteriormente integrado, validado e armazenado de forma independente, permitindo retomar o processamento sem repetir os lotes concluídos.

Nesta etapa, nenhuma junção entre os conjuntos de dados é realizada.

In [ ]:
# 14. PREPARAÇÃO DA INTEGRAÇÃO NACIONAL

import math

# Diretórios temporários
pasta_integracao = os.path.join(
    pasta_local, "INTEGRACAO_NACIONAL"
)

pasta_entrada = os.path.join(
    pasta_integracao, "ENTRADA"
)

pasta_temporaria = os.path.join(
    pasta_integracao, "TEMP"
)

# Diretório persistente dos lotes
pasta_lotes = os.path.join(
    pasta_resultados, "LOTES_INTEGRACAO"
)

for caminho in [
    pasta_integracao,
    pasta_entrada,
    pasta_temporaria,
    pasta_lotes
]:
    os.makedirs(caminho, exist_ok=True)

# Verificar espaço disponível
espaco_livre = shutil.disk_usage(
    pasta_local
).free

tamanho_arquivos = sum(
    os.path.getsize(caminhos_parquet[tema])
    for tema in temas_banco
)

if espaco_livre < tamanho_arquivos * 4:
    raise RuntimeError(
        "Espaço em disco insuficiente para "
        "a preparação da integração."
    )

# Copiar arquivos para o disco local do Colab
caminhos_locais = {}

print("PREPARAÇÃO DOS ARQUIVOS")
print("-" * 55)

for tema in temas_banco:

    origem = caminhos_parquet[tema]

    destino = os.path.join(
        pasta_entrada,
        f"{tema}_2022.parquet"
    )

    # Reutilizar cópia local somente se o tamanho coincidir
    if (
        not os.path.isfile(destino)
        or os.path.getsize(destino) != os.path.getsize(origem)
    ):
        shutil.copy2(origem, destino)

    # Conferir estrutura da cópia
    parquet = pq.ParquetFile(destino)

    if parquet.metadata.num_rows != len(
        setores_por_tema[tema]
    ):
        raise ValueError(
            f"{tema}: cópia local inconsistente."
        )

    caminhos_locais[tema] = destino

    print(f"{tema}: arquivo local disponível.")

# Configurar DuckDB
con = duckdb.connect(":memory:")

con.execute("SET threads = 2")
con.execute("SET memory_limit = '4GB'")

con.execute(
    f"SET temp_directory = '{pasta_temporaria}'"
)

con.execute(
    "SET preserve_insertion_order = false"
)

# Obter os setores do arquivo Básico
setores_ordenados = con.execute(
    """
    SELECT CD_SETOR
    FROM read_parquet(?)
    ORDER BY CD_SETOR
    """,
    [caminhos_locais["BASICO"]]
).fetchnumpy()["CD_SETOR"]

# Definir tamanho e quantidade de lotes
tamanho_lote = 10000

total_setores = len(setores_ordenados)

quantidade_lotes = math.ceil(
    total_setores / tamanho_lote
)

# Construir o plano de processamento
plano_lotes = []

for indice in range(quantidade_lotes):

    inicio = indice * tamanho_lote

    fim = min(
        inicio + tamanho_lote,
        total_setores
    )

    plano_lotes.append({
        "Lote": indice + 1,
        "Primeiro setor": setores_ordenados[inicio],
        "Último setor": setores_ordenados[fim - 1],
        "Quantidade": fim - inicio
    })

df_plano_lotes = pd.DataFrame(plano_lotes)

# Validar o plano
assert total_setores == 468099

assert (
    df_plano_lotes["Quantidade"].sum()
    == total_setores
)

assert quantidade_lotes == 47

# Registrar o plano no Google Drive
caminho_plano = os.path.join(
    pasta_resultados,
    "PLANO_INTEGRACAO_2022.csv"
)

df_plano_lotes.to_csv(
    caminho_plano,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("\nPLANO DE INTEGRAÇÃO NACIONAL")
print("-" * 55)

print("Setores:", total_setores)
print("Setores por lote:", tamanho_lote)
print("Quantidade de lotes:", quantidade_lotes)
print("Total planejado:",
      df_plano_lotes["Quantidade"].sum())

print("\nPrimeiros lotes:")
display(df_plano_lotes.head())

print("\nÚltimo lote:")
display(df_plano_lotes.tail(1))

print("\nPlano de integração preparado!")


*15. Teste de integração nacional - Primeira parte*

Realizar um teste de integração dos oito conjuntos de dados do Censo Demográfico 2022, utilizando os primeiros 10.000 setores censitários definidos no plano de processamento.

O arquivo Básico constitui a referência territorial. As variáveis dos sete conjuntos temáticos são incorporadas por meio de junções à esquerda (`LEFT JOIN`), utilizando exclusivamente o identificador padronizado `CD_SETOR`.

Para reduzir o consumo de memória, cada conjunto é filtrado pelos limites territoriais do lote antes da integração.

O resultado é armazenado em formato Parquet e submetido a verificações de quantidade de registros, estrutura das colunas, unicidade dos identificadores e correspondência dos dados.

A validação deste lote permitirá avaliar a estratégia antes da execução do processamento nacional completo.

In [ ]:
# 15. TESTE DE INTEGRAÇÃO - PRIMEIRA PARTE

# Selecionar a primeira parte do plano
lote = df_plano_lotes.iloc[0]

primeiro_setor = str(lote["Primeiro setor"])
ultimo_setor = str(lote["Último setor"])
quantidade_esperada = int(lote["Quantidade"])

# Definir os arquivos de saída
saida_teste = os.path.join(
    pasta_integracao,
    "TESTE_LOTE_001.parquet"
)

saida_drive = os.path.join(
    pasta_lotes,
    "TESTE_LOTE_001.parquet"
)

# Proteger caminhos utilizados nas consultas SQL
def literal_sql(valor):
    return "'" + str(valor).replace("'", "''") + "'"

# Construir referências dos oito conjuntos
aliases = {
    "BASICO": "b",
    "DOM1": "d1",
    "DOM2": "d2",
    "DOM3": "d3",
    "DEM": "dem",
    "PAR": "par",
    "COR": "cor",
    "ALF": "alf"
}

temas_adicionais = [
    tema for tema in temas_banco
    if tema != "BASICO"
]

# Filtrar cada conjunto antes da junção
consultas_filtradas = {}

for tema, alias in aliases.items():

    caminho = literal_sql(caminhos_locais[tema])

    consultas_filtradas[tema] = (
        f"(SELECT * FROM read_parquet({caminho}) "
        f"WHERE CD_SETOR BETWEEN "
        f"{literal_sql(primeiro_setor)} AND "
        f"{literal_sql(ultimo_setor)}) {alias}"
    )

# Preservar todas as colunas do Básico
colunas_select = ["b.*"]

# Incorporar as variáveis temáticas sem repetir CD_SETOR
for tema in temas_adicionais:
    alias = aliases[tema]
    colunas_select.append(
        f"{alias}.* EXCLUDE (CD_SETOR)"
    )

# Construir a consulta de integração
sql_integracao = (
    "SELECT\n    "
    + ",\n    ".join(colunas_select)
    + "\nFROM "
    + consultas_filtradas["BASICO"]
)

for tema in temas_adicionais:
    sql_integracao += (
        "\nLEFT JOIN "
        + consultas_filtradas[tema]
        + " USING (CD_SETOR)"
    )

# Exportar o lote integrado para Parquet
sql_exportacao = (
    f"COPY ({sql_integracao}) "
    f"TO {literal_sql(saida_teste)} "
    "(FORMAT PARQUET, COMPRESSION ZSTD)"
)

print("TESTE DE INTEGRAÇÃO NACIONAL")
print("-" * 60)
print("Lote: 1")
print("Primeiro setor:", primeiro_setor)
print("Último setor:", ultimo_setor)
print("Setores esperados:", quantidade_esperada)

print("\nExecutando a integração...")

con.execute(sql_exportacao)

# Verificar a estrutura do arquivo gerado
metadados = pq.ParquetFile(saida_teste)

linhas = metadados.metadata.num_rows
nomes_colunas = metadados.schema_arrow.names
colunas = len(nomes_colunas)

print("\nVALIDAÇÃO ESTRUTURAL")
print("-" * 60)

print("Registros:", linhas)
print("Colunas:", colunas)

assert linhas == quantidade_esperada
assert colunas == 1356
assert len(nomes_colunas) == len(set(nomes_colunas))

# Validar identificadores e unicidade
total, distintos = con.execute(
    """
    SELECT
        COUNT(*),
        COUNT(DISTINCT CD_SETOR)
    FROM read_parquet(?)
    """,
    [saida_teste]
).fetchone()

assert total == quantidade_esperada
assert distintos == quantidade_esperada

# Comparar os códigos do lote com o arquivo Básico
sql_comparacao = f"""
WITH esperado AS (
    SELECT CD_SETOR
    FROM read_parquet(
        {literal_sql(caminhos_locais["BASICO"])}
    )
    WHERE CD_SETOR BETWEEN
        {literal_sql(primeiro_setor)}
        AND {literal_sql(ultimo_setor)}
),
obtido AS (
    SELECT CD_SETOR
    FROM read_parquet({literal_sql(saida_teste)})
)
SELECT
    (
        SELECT COUNT(*)
        FROM (
            SELECT CD_SETOR FROM esperado
            EXCEPT
            SELECT CD_SETOR FROM obtido
        )
    )
    +
    (
        SELECT COUNT(*)
        FROM (
            SELECT CD_SETOR FROM obtido
            EXCEPT
            SELECT CD_SETOR FROM esperado
        )
    )
"""

diferencas = con.execute(
    sql_comparacao
).fetchone()[0]

assert diferencas == 0, (
    "Existem diferenças entre os setores "
    "do lote e os setores do Básico."
)

print("Setores distintos:", distintos)
print("Diferenças territoriais:", diferencas)

# Conferir uma variável de cada conjunto temático
print("\nVALIDAÇÃO DA CORRESPONDÊNCIA TEMÁTICA")
print("-" * 60)

for tema in temas_adicionais:

    # Selecionar a primeira variável temática
    variavel = [
        nome for nome in esquemas[tema].values()
        if nome != "CD_SETOR"
    ][0]

    origem = literal_sql(caminhos_locais[tema])
    integrado = literal_sql(saida_teste)

    sql_verificacao = f"""
        SELECT COUNT(*)
        FROM read_parquet({integrado}) b
        LEFT JOIN read_parquet({origem}) t
            USING (CD_SETOR)
        WHERE b."{variavel}"
            IS DISTINCT FROM t."{variavel}"
    """

    divergencias = con.execute(
        sql_verificacao
    ).fetchone()[0]

    assert divergencias == 0, (
        f"{tema}: divergências na variável {variavel}."
    )

    print(f"{tema}: correspondência validada.")

# Salvar o lote de teste no Drive
saida_parcial = saida_drive + ".parcial"

shutil.copy2(saida_teste, saida_parcial)

copia = pq.ParquetFile(saida_parcial)

assert copia.metadata.num_rows == quantidade_esperada
assert len(copia.schema_arrow.names) == 1356

os.replace(saida_parcial, saida_drive)

tamanho_mb = os.path.getsize(
    saida_drive
) / (1024**2)

print("\nRESULTADO DO TESTE")
print("-" * 60)

print("Setores integrados:", linhas)
print("Colunas:", colunas)
print("Setores únicos:", distintos)
print("Tamanho:", f"{tamanho_mb:.2f} MB")
print("Arquivo:", saida_drive)

print("\nPRIMEIRO LOTE INTEGRADO E VALIDADO!")


*16. Integração automatizada dos lotes censitários*

Automatizar a integração dos conjuntos de dados do Censo Demográfico 2022 a partir dos lotes definidos pelo arquivo Básico.

Para cada lote, os oito conjuntos são filtrados pelos limites de `CD_SETOR` antes da realização das junções à esquerda (`LEFT JOIN`). O procedimento preserva os setores do Básico e incorpora as variáveis dos sete conjuntos temáticos.

Os arquivos integrados são validados quanto à quantidade de registros, ao número de colunas, à unicidade dos identificadores e à correspondência com os limites territoriais definidos.

Cada lote é salvo individualmente no Google Drive. Arquivos previamente concluídos somente são reutilizados após validação, permitindo retomar o processamento em caso de interrupção.

Inicialmente, serão processados três lotes para verificar o funcionamento da rotina automatizada antes de sua aplicação ao conjunto nacional.

In [ ]:
# 16. INTEGRAÇÃO AUTOMATIZADA COM RETOMADA

import time

# Teste inicial: processar apenas os três primeiros lotes
numero_lotes_teste = 3

# Colunas esperadas no banco integrado
colunas_esperadas = list(
    esquemas["BASICO"].values()
)

for tema in temas_adicionais:
    colunas_esperadas.extend([
        nome for nome in esquemas[tema].values()
        if nome != "CD_SETOR"
    ])

assert len(colunas_esperadas) == 1356


def validar_lote(caminho, lote):
    """Verifica a estrutura e os setores de um lote integrado."""

    if not os.path.isfile(caminho):
        return False

    try:
        parquet = pq.ParquetFile(caminho)

        if parquet.schema_arrow.names != colunas_esperadas:
            return False

        esperado = int(lote["Quantidade"])

        if parquet.metadata.num_rows != esperado:
            return False

        total, distintos, primeiro, ultimo = con.execute(
            """
            SELECT
                COUNT(*),
                COUNT(DISTINCT CD_SETOR),
                MIN(CD_SETOR),
                MAX(CD_SETOR)
            FROM read_parquet(?)
            """,
            [caminho]
        ).fetchone()

        return (
            total == esperado
            and distintos == esperado
            and primeiro == str(lote["Primeiro setor"])
            and ultimo == str(lote["Último setor"])
        )

    except Exception:
        return False


def integrar_lote(lote):
    """Integra os oito conjuntos para um intervalo de setores."""

    numero = int(lote["Lote"])
    primeiro = str(lote["Primeiro setor"])
    ultimo = str(lote["Último setor"])

    nome = f"LOTE_{numero:03d}_2022.parquet"

    destino = os.path.join(pasta_lotes, nome)

    # Retomar sem repetir arquivos válidos
    if validar_lote(destino, lote):
        return "REUTILIZADO"

    # Reaproveitar o teste validado da Célula 15
    if numero == 1:

        arquivo_teste = os.path.join(
            pasta_lotes, "TESTE_LOTE_001.parquet"
        )

        if validar_lote(arquivo_teste, lote):

            parcial = destino + ".parcial"

            shutil.copy2(arquivo_teste, parcial)

            if not validar_lote(parcial, lote):
                raise ValueError(
                    "Falha na validação da cópia do lote 1."
                )

            os.replace(parcial, destino)
            return "REUTILIZADO DO TESTE"

    # Caminho temporário local
    local = os.path.join(pasta_integracao, nome)

    # Montar consultas com filtragem anterior à junção
    consultas = {}

    for tema, alias in aliases.items():

        caminho = literal_sql(caminhos_locais[tema])

        consultas[tema] = (
            f"(SELECT * FROM read_parquet({caminho}) "
            f"WHERE CD_SETOR BETWEEN "
            f"{literal_sql(primeiro)} AND "
            f"{literal_sql(ultimo)}) {alias}"
        )

    selecao = ["b.*"]

    for tema in temas_adicionais:
        selecao.append(
            f"{aliases[tema]}.* EXCLUDE (CD_SETOR)"
        )

    sql = (
        "SELECT " + ", ".join(selecao)
        + " FROM " + consultas["BASICO"]
    )

    for tema in temas_adicionais:
        sql += (
            " LEFT JOIN " + consultas[tema]
            + " USING (CD_SETOR)"
        )

    sql_copy = (
        f"COPY ({sql}) TO {literal_sql(local)} "
        "(FORMAT PARQUET, COMPRESSION ZSTD)"
    )

    parcial = destino + ".parcial"

    try:
        # Integração no disco local
        con.execute(sql_copy)

        if not validar_lote(local, lote):
            raise ValueError(
                f"Lote {numero}: resultado local inválido."
            )

        # Salvar e validar cópia no Drive
        shutil.copy2(local, parcial)

        if not validar_lote(parcial, lote):
            raise ValueError(
                f"Lote {numero}: cópia no Drive inválida."
            )

        os.replace(parcial, destino)

        return "INTEGRADO E VALIDADO"

    finally:
        # Excluir somente arquivos temporários
        for caminho in [local, parcial]:
            if os.path.isfile(caminho):
                os.remove(caminho)


# EXECUÇÃO CONTROLADA


print("INTEGRAÇÃO NACIONAL - EXECUÇÃO DE TESTE")
print("-" * 60)

resultados_lotes = []

for _, lote in df_plano_lotes.head(
    numero_lotes_teste
).iterrows():

    numero = int(lote["Lote"])

    inicio = time.perf_counter()

    status = integrar_lote(lote)

    duracao = time.perf_counter() - inicio

    resultados_lotes.append({
        "Lote": numero,
        "Setores": int(lote["Quantidade"]),
        "Status": status,
        "Tempo (s)": round(duracao, 2)
    })

    print(
        f"Lote {numero:03d}: {status} "
        f"| {duracao:.2f} segundos",
        flush=True
    )

# Resumo dos três primeiros lotes
df_resultados_lotes = pd.DataFrame(resultados_lotes)

print("\nRESUMO DO TESTE AUTOMATIZADO")
print("-" * 60)

display(df_resultados_lotes)

print(
    "Setores contemplados:",
    df_resultados_lotes["Setores"].sum()
)

print("\nTeste automatizado concluído!")


*17. Execução da integração nacional*

Executar a integração dos 47 lotes definidos pelo arquivo Básico do Censo Demográfico 2022.

O processamento utiliza a rotina validada anteriormente, preservando os 468.099 setores censitários de referência e incorporando as variáveis dos sete conjuntos temáticos.

Cada lote é validado e salvo individualmente no Google Drive. Lotes previamente concluídos e válidos são reutilizados, permitindo a retomada do processamento em caso de interrupção.

O resultado desta etapa corresponde a um conjunto de arquivos Parquet integrados, que serão posteriormente reunidos em um único banco nacional por setor censitário.

In [ ]:
# 17. EXECUÇÃO DA INTEGRAÇÃO NACIONAL

import time

print("INTEGRAÇÃO NACIONAL - CENSO 2022")
print("-" * 65)

resultados_nacionais = []

for _, lote in df_plano_lotes.iterrows():

    numero = int(lote["Lote"])

    inicio = time.perf_counter()

    # Utilizar a função validada na Célula 16
    status = integrar_lote(lote)

    duracao = time.perf_counter() - inicio

    resultados_nacionais.append({
        "Lote": numero,
        "Setores": int(lote["Quantidade"]),
        "Status": status,
        "Tempo (s)": round(duracao, 2)
    })

    print(
        f"Lote {numero:03d}/047: {status} "
        f"| {duracao:.2f} s",
        flush=True
    )

# Organizar os resultados
df_resultados_nacionais = pd.DataFrame(
    resultados_nacionais
)

# Verificar a quantidade total de setores
total_integrado = (
    df_resultados_nacionais["Setores"].sum()
)

assert len(df_resultados_nacionais) == 47

assert total_integrado == 468099

print("\nRESUMO DO PROCESSAMENTO")
print("-" * 65)

print("Lotes concluídos:", len(df_resultados_nacionais))
print("Setores contemplados:", total_integrado)

print(
    "Tempo total:",
    f"{df_resultados_nacionais['Tempo (s)'].sum():.2f} s"
)

display(df_resultados_nacionais.tail(10))

print("\nINTEGRAÇÃO DOS LOTES CONCLUÍDA!")


*18. Consolidação do banco nacional por setor censitário*

Esta etapa reúne os 47 lotes previamente integrados em um único arquivo Parquet de abrangência nacional.

A consolidação é realizada com a biblioteca PyArrow, por meio da leitura sequencial dos arquivos em pequenos blocos, para limitar o consumo de memória RAM.

O procedimento preserva todas as variáveis e seus valores originais, sem realizar novas junções ou agregações territoriais.

Após a consolidação, são verificados o número de registros, a estrutura das colunas, a unicidade dos identificadores e a correspondência integral dos setores censitários com o arquivo Básico do IBGE.

O banco nacional é armazenado no Google Drive para posterior validação temática e utilização nas análises da pesquisa.

In [ ]:
# 18A. RECUPERAÇÃO DOS LOTES APÓS REINÍCIO

import os
import pyarrow.parquet as pq

from google.colab import drive

# Reconectar o Google Drive
drive.mount("/content/drive")

# Diretório principal do projeto
# Recuperar o diretório configurado na Célula 4
if "pasta_drive" not in globals():
    raise RuntimeError(
        "Execute a Célula 4 antes de recuperar os lotes."
    )

# Recuperar os diretórios
pasta_resultados = os.path.join(
    pasta_drive, "RESULTADOS_2022"
)

pasta_lotes = os.path.join(
    pasta_resultados, "LOTES_INTEGRACAO"
)

pasta_local = "/content/censo2022"

os.makedirs(pasta_local, exist_ok=True)

# Conferir os 47 arquivos de integração
lotes_encontrados = []
lotes_ausentes = []
total_setores = 0

print("RECUPERAÇÃO DOS LOTES")
print("-" * 55)

for numero in range(1, 48):

    nome = f"LOTE_{numero:03d}_2022.parquet"
    caminho = os.path.join(pasta_lotes, nome)

    if not os.path.isfile(caminho):
        lotes_ausentes.append(nome)
        continue

    parquet = pq.ParquetFile(caminho)

    if len(parquet.schema_arrow.names) != 1356:
        raise ValueError(
            f"{nome}: quantidade de colunas inesperada."
        )

    total_setores += parquet.metadata.num_rows
    lotes_encontrados.append(caminho)

print("Lotes encontrados:", len(lotes_encontrados))
print("Lotes ausentes:", len(lotes_ausentes))
print("Total de registros:", total_setores)

if lotes_ausentes:
    print("\nArquivos ausentes:")
    for nome in lotes_ausentes:
        print(nome)

assert len(lotes_encontrados) == 47
assert total_setores == 468099

print("\nArquivos recuperados com sucesso!")


In [ ]:
# 18B. CONSOLIDAÇÃO NACIONAL EM BLOCOS - PYARROW

import os
import shutil
import pyarrow.parquet as pq

# 1. Definir os arquivos de entrada e saída
nome_nacional = "UNIVERSO_2022_NACIONAL.parquet"

saida_local = os.path.join(
    pasta_local, nome_nacional
)

saida_drive = os.path.join(
    pasta_resultados, nome_nacional
)

saida_parcial = saida_drive + ".parcial"

# Arquivo Básico para validação territorial
caminho_basico = os.path.join(
    pasta_drive,
    "DADOS_TRATADOS_2022",
    "BASICO_2022.parquet"
)

# 2. Conferir os lotes antes de consolidar
assert len(lotes_encontrados) == 47
assert os.path.isfile(caminho_basico)

schema_referencia = pq.ParquetFile(
    lotes_encontrados[0]
).schema_arrow

assert len(schema_referencia.names) == 1356

# Conferir se todos possuem o mesmo esquema
for numero, caminho in enumerate(
    lotes_encontrados, start=1
):
    parquet = pq.ParquetFile(caminho)

    assert parquet.schema_arrow.equals(
        schema_referencia
    ), f"Esquema divergente no lote {numero}"

# 3. Consolidar sequencialmente
total_linhas = 0

print("CONSOLIDAÇÃO NACIONAL EM BLOCOS")
print("-" * 60)

with pq.ParquetWriter(
    saida_local,
    schema=schema_referencia,
    compression="zstd"
) as escritor:

    for numero, caminho in enumerate(
        lotes_encontrados, start=1
    ):

        parquet = pq.ParquetFile(caminho)
        linhas_lote = 0

        # Ler um pequeno bloco por vez
        for bloco in parquet.iter_batches(
            batch_size=1000,
            use_threads=False
        ):
            escritor.write_batch(bloco)
            linhas_lote += bloco.num_rows

        # Validar quantidade do lote
        esperado = 10000 if numero < 47 else 8099

        assert linhas_lote == esperado, (
            f"Lote {numero}: registros divergentes."
        )

        total_linhas += linhas_lote

        print(
            f"Lote {numero:03d}/047: "
            f"{linhas_lote:,} registros | "
            f"Acumulado: {total_linhas:,}",
            flush=True
        )

# 4. Validar o banco nacional
print("\nVALIDAÇÃO DO BANCO CONSOLIDADO")
print("-" * 60)

parquet_final = pq.ParquetFile(saida_local)

assert parquet_final.metadata.num_rows == 468099
assert parquet_final.schema_arrow.equals(
    schema_referencia
)

# Conferir todos os identificadores
setores_finais = set()
registros_verificados = 0

for bloco in parquet_final.iter_batches(
    batch_size=10000,
    columns=["CD_SETOR"]
):
    codigos = bloco.column(0).to_pylist()
    registros_verificados += len(codigos)

    setores_finais.update(codigos)

assert registros_verificados == 468099
assert len(setores_finais) == 468099

# Conferir os setores do arquivo Básico
setores_basico = set()

parquet_basico = pq.ParquetFile(caminho_basico)

for bloco in parquet_basico.iter_batches(
    batch_size=10000,
    columns=["CD_SETOR"]
):
    setores_basico.update(
        bloco.column(0).to_pylist()
    )

assert setores_finais == setores_basico, (
    "O banco nacional não corresponde "
    "aos setores do arquivo Básico."
)

print("Registros:", registros_verificados)
print("Setores distintos:", len(setores_finais))
print("Colunas:", len(schema_referencia.names))
print("Correspondência com o Básico: OK")

# 5. Salvar no Google Drive
print("\nSalvando o banco nacional...")

shutil.copy2(saida_local, saida_parcial)

# Conferir a estrutura da cópia
copia = pq.ParquetFile(saida_parcial)

assert copia.metadata.num_rows == 468099
assert copia.schema_arrow.equals(schema_referencia)

# Finalizar o arquivo no Drive
os.replace(saida_parcial, saida_drive)

tamanho_mb = (
    os.path.getsize(saida_drive) / (1024**2)
)

print("\nBANCO NACIONAL CONSOLIDADO")
print("-" * 60)

print("Registros:", registros_verificados)
print("Colunas:", len(schema_referencia.names))
print("Setores únicos:", len(setores_finais))
print("Tamanho:", f"{tamanho_mb:.2f} MB")
print("Arquivo:", saida_drive)

print("\nCONSOLIDAÇÃO NACIONAL CONCLUÍDA!")


*19. Validação da cobertura temática do banco nacional*

Verificar a a correspondência territorial entre o banco nacional consolidado e os arquivos temáticos originais do IBGE.

São identificados os setores presentes exclusivamente no arquivo Básico e verificada a ausência de informações temáticas para esses registros. Também é conferida a presença das informações para os setores contemplados pelos conjuntos temáticos.

A validação distingue valores ausentes decorrentes da inexistência de registros temáticos dos valores protegidos por sigilo estatístico, identificados por X. Nenhum valor é modificado durante o procedimento.

In [ ]:
# 19. VALIDAÇÃO DA COBERTURA TEMÁTICA

import os
import pyarrow.parquet as pq

# Arquivos para comparação
caminho_nacional = os.path.join(
    pasta_resultados,
    "UNIVERSO_2022_NACIONAL.parquet"
)

caminho_dom1 = os.path.join(
    pasta_drive,
    "DADOS_TRATADOS_2022",
    "DOM1_2022.parquet"
)

# Identificar setores presentes no conjunto temático
setores_tematicos = set()

arquivo_dom1 = pq.ParquetFile(caminho_dom1)

for bloco in arquivo_dom1.iter_batches(
    batch_size=10000,
    columns=["CD_SETOR"]
):
    setores_tematicos.update(
        bloco.column(0).to_pylist()
    )

# Verificar os registros no banco nacional
arquivo_nacional = pq.ParquetFile(caminho_nacional)

total_basico_exclusivo = 0
total_com_tema = 0
ausencias_inesperadas = 0
valores_inesperados = 0
total_x = 0

# Primeira variável do conjunto DOM1
variavel_teste = "DOM1_V00001"

for bloco in arquivo_nacional.iter_batches(
    batch_size=10000,
    columns=["CD_SETOR", variavel_teste]
):
    codigos = bloco.column(0).to_pylist()
    valores = bloco.column(1).to_pylist()

    for codigo, valor in zip(codigos, valores):

        if codigo not in setores_tematicos:
            total_basico_exclusivo += 1

            if valor is not None:
                valores_inesperados += 1

        else:
            total_com_tema += 1

            if valor is None:
                ausencias_inesperadas += 1

        if valor == "X":
            total_x += 1

# Exibir os resultados
print("VALIDAÇÃO DA COBERTURA TEMÁTICA")
print("-" * 60)

print("Setores com correspondência temática:", total_com_tema)
print("Setores exclusivos do Básico:", total_basico_exclusivo)
print("Ausências inesperadas:", ausencias_inesperadas)
print("Valores inesperados:", valores_inesperados)
print(f"Valores X em {variavel_teste}:", total_x)

# Verificações
assert total_com_tema == 458772
assert total_basico_exclusivo == 9327
assert ausencias_inesperadas == 0
assert valores_inesperados == 0

print("\nCobertura temática validada!")


*20. Visualização do banco nacional integrado*

Esta etapa permite explorar o banco nacional do Censo Demográfico 2022 após sua consolidação.

O arquivo Parquet é consultado por meio do DuckDB, permitindo visualizar uma amostra dos registros e examinar a estrutura das variáveis sem carregar integralmente o banco na memória.

A visualização é utilizada para conferir a organização das informações por setor censitário, preservando os dados originais e a estrutura territorial do arquivo Básico.

In [ ]:
# 20. VISUALIZAÇÃO DO BANCO NACIONAL

import duckdb
import os

# Localizar o banco nacional
caminho_nacional = os.path.join(
    pasta_resultados,
    "UNIVERSO_2022_NACIONAL.parquet"
)

# Abrir uma conexão com o DuckDB
con_visualizacao = duckdb.connect(":memory:")

# Criar uma visualização lógica, sem copiar os dados
con_visualizacao.execute(
    """
    CREATE VIEW censo2022 AS
    SELECT *
    FROM read_parquet(?)
    """.replace(
        "read_parquet(?)",
        f"read_parquet('{caminho_nacional}')"
    )
)

# Consultar os primeiros 20 setores e 15 colunas
colunas_amostra = list(
    con_visualizacao.execute(
        "DESCRIBE censo2022"
    ).df()["column_name"].head(15)
)

colunas_sql = ", ".join(
    f'"{coluna}"'
    for coluna in colunas_amostra
)

df_visualizacao = con_visualizacao.execute(
    f"""
    SELECT {colunas_sql}
    FROM censo2022
    ORDER BY CD_SETOR
    LIMIT 20
    """
).df()

print("VISUALIZAÇÃO DO BANCO NACIONAL")
print("-" * 60)

print("Exibindo 20 setores e 15 colunas.")
print("O banco completo permanece no arquivo Parquet.")

display(df_visualizacao)


In [ ]:

# Visualizar variáveis de diferentes conjuntos temáticos

df_exemplo = con_visualizacao.execute("""
    SELECT
        CD_SETOR,
        SITUACAO,
        v0001,
        DOM1_V00001,
        DEM_V01006,
        PAR_V01042,
        COR_V01317,
        ALF_V00644
    FROM censo2022
    ORDER BY CD_SETOR
    LIMIT 20
""").df()

display(df_exemplo)


In [ ]:
# BACKUP DO BANCO NACIONAL - CENSO 2022

import os
import shutil
import hashlib

# Utilizar o diretório configurado na seção 4
raiz = pasta_drive

# Arquivo original
origem = os.path.join(
    raiz, "RESULTADOS_2022",
    "UNIVERSO_2022_NACIONAL.parquet"
)

# Pasta de backup separada dos resultados
pasta_backup = os.path.join(
    raiz, "BACKUP_BANCO_NACIONAL"
)

os.makedirs(pasta_backup, exist_ok=True)

destino = os.path.join(
    pasta_backup,
    "UNIVERSO_2022_NACIONAL_BACKUP.parquet"
)

# Calcular assinatura digital do arquivo
def calcular_sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as arquivo:
        for bloco in iter(
            lambda: arquivo.read(8 * 1024 * 1024),
            b""
        ):
            h.update(bloco)
    return h.hexdigest()

if not os.path.isfile(origem):
    raise FileNotFoundError(origem)

hash_original = calcular_sha256(origem)

# Não sobrescrever um backup existente diferente
if os.path.exists(destino):
    hash_backup = calcular_sha256(destino)
    if hash_backup != hash_original:
        raise FileExistsError(
            "Já existe um backup diferente. "
            "O arquivo existente foi preservado."
        )
    print("Backup idêntico já existente.")
else:
    temporario = destino + ".parcial"

    shutil.copy2(origem, temporario)

    if calcular_sha256(temporario) != hash_original:
        raise ValueError("A cópia não passou na validação.")

    os.replace(temporario, destino)
    print("Backup criado com sucesso!")

# Conferência final
assert calcular_sha256(destino) == hash_original

print("Integridade: CONFIRMADA")
print("SHA-256:", hash_original)
print("Backup:", destino)


*21. Validação integral das variáveis do banco nacional*

Comparar os valores das variáveis do banco nacional integrado com os respectivos arquivos Parquet de origem, utilizando CD_SETOR como chave de correspondência. O objetivo é verificar a integridade dos dados após a integração, identificar eventuais divergências e registrar os resultados da validação.

In [9]:
#VALIDAÇÃO INTEGRAL DAS VARIÁVEIS DO BANCO NACIONAL

import os
import glob
import pyarrow.parquet as pq

# Recuperar os diretórios configurados no início do notebook
pasta_tratados = os.path.join(
    pasta_drive, "DADOS_TRATADOS_2022"
)

pasta_resultados = os.path.join(
    pasta_drive, "RESULTADOS_2022"
)

arquivo_nacional = os.path.join(
    pasta_resultados, "UNIVERSO_2022_NACIONAL.parquet"
)

# Identificar os arquivos para a validação
arquivos_origem = sorted(
    glob.glob(os.path.join(pasta_tratados, "*.parquet"))
)

print("PREPARAÇÃO DA VALIDAÇÃO")
print("-" * 50)

print("Banco nacional:", os.path.exists(arquivo_nacional))
print("Arquivos de origem encontrados:", len(arquivos_origem))

for arquivo in arquivos_origem:
    metadata = pq.ParquetFile(arquivo)
    print(
        os.path.basename(arquivo),
        "| Linhas:", metadata.metadata.num_rows,
        "| Colunas:", len(metadata.schema_arrow.names)
    )

if os.path.exists(arquivo_nacional):
    metadata = pq.ParquetFile(arquivo_nacional)
    print(
        "\nBanco nacional:",
        metadata.metadata.num_rows, "linhas |",
        len(metadata.schema_arrow.names), "colunas"
    )


PREPARAÇÃO DA VALIDAÇÃO
--------------------------------------------------
Banco nacional: True
Arquivos de origem encontrados: 8
ALF_2022.parquet | Linhas: 458772 | Colunas: 363
BASICO_2022.parquet | Linhas: 468099 | Colunas: 38
COR_2022.parquet | Linhas: 458772 | Colunas: 96
DEM_2022.parquet | Linhas: 458772 | Colunas: 37
DOM1_2022.parquet | Linhas: 458772 | Colunas: 90
DOM2_2022.parquet | Linhas: 458772 | Colunas: 407
DOM3_2022.parquet | Linhas: 458772 | Colunas: 149
PAR_2022.parquet | Linhas: 458772 | Colunas: 183

Banco nacional: 468099 linhas | 1356 colunas


*21.2. Teste inicial de correspondência dos valores*

Compara dez variáveis do conjunto DOM1 entre o arquivo de origem e o banco nacional, utilizando o código do setor censitário como chave. A comparação distingue valores nulos, valores suprimidos (X) e demais representações originais.

In [11]:
#TESTE INICIAL DE CORRESPONDÊNCIA DOS VALORES

import os
import duckdb

# Arquivo original do conjunto DOM1
arquivo_dom1 = os.path.join(
    pasta_tratados, "DOM1_2022.parquet"
)

# Conexão DuckDB para consulta
con = duckdb.connect()

# Recuperar nomes das variáveis originais
colunas_dom1 = [
    campo.name
    for campo in pq.ParquetFile(arquivo_dom1).schema_arrow
    if campo.name != "CD_SETOR"
]

# Selecionar as primeiras 10 variáveis
variaveis_teste = colunas_dom1[:10]

# Escapar identificadores SQL
def identificar(nome):
    return '"' + nome.replace('"', '""') + '"'

# Comparar cada variável com sua equivalente no banco nacional
expressoes = []

for variavel in variaveis_teste:
    nome_nacional = variavel

    expressoes.append(
        f"""
        COUNT(*) FILTER (
            WHERE o.{identificar(variavel)}
                  IS DISTINCT FROM
                  n.{identificar(nome_nacional)}
        ) AS {identificar(variavel)}
        """
    )

consulta = f"""
SELECT
    {", ".join(expressoes)}
FROM read_parquet(?) AS o
LEFT JOIN read_parquet(?) AS n
    ON o.CD_SETOR = n.CD_SETOR
"""

resultado = con.execute(
    consulta,
    [arquivo_dom1, arquivo_nacional]
).fetchdf()

print("VALIDAÇÃO INICIAL — DOM1")
print("-" * 50)

total_divergencias = 0

for variavel in variaveis_teste:
    divergencias = int(resultado.loc[0, variavel])
    total_divergencias += divergencias

    print(
        f"{variavel}: {divergencias:,} divergências"
    )

print("-" * 50)
print("TOTAL DE DIVERGÊNCIAS:", total_divergencias)

if total_divergencias == 0:
    print("RESULTADO: 10 variáveis sem divergências.")
else:
    print("ATENÇÃO: foram identificadas divergências.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

VALIDAÇÃO INICIAL — DOM1
--------------------------------------------------
DOM1_V00001: 0 divergências
DOM1_V00002: 0 divergências
DOM1_V00003: 0 divergências
DOM1_V00004: 0 divergências
DOM1_V00005: 0 divergências
DOM1_V00006: 0 divergências
DOM1_V00007: 0 divergências
DOM1_V00008: 0 divergências
DOM1_V00009: 0 divergências
DOM1_V00010: 0 divergências
--------------------------------------------------
TOTAL DE DIVERGÊNCIAS: 0
RESULTADO: 10 variáveis sem divergências.


*21.3. Validação integral das variáveis do conjunto DOM1*

Comparar todas as variáveis do conjunto DOM1 com as respectivas colunas do banco nacional. O processamento ocorre em blocos de 20 variáveis para controlar a utilização de memória. Os resultados permitem identificar divergências nos valores, inclusive diferenças entre valores nulos e códigos de supressão estatística.

In [12]:
# VALIDAÇÃO INTEGRAL DAS VARIÁVEIS DO CONJUNTO DOM1

import pandas as pd

# Todas as variáveis do DOM1, exceto a chave territorial
variaveis_dom1 = [
    nome
    for nome in pq.ParquetFile(arquivo_dom1).schema_arrow.names
    if nome != "CD_SETOR"
]

# Tamanho do bloco de comparação
tamanho_bloco = 20

resultados_dom1 = []

print("VALIDAÇÃO INTEGRAL — DOM1")
print("-" * 55)

for inicio in range(0, len(variaveis_dom1), tamanho_bloco):

    bloco = variaveis_dom1[inicio:inicio + tamanho_bloco]

    expressoes = [
        f"""
        COUNT(*) FILTER (
            WHERE o.{identificar(nome)}
            IS DISTINCT FROM n.{identificar(nome)}
        ) AS {identificar(nome)}
        """
        for nome in bloco
    ]

    consulta = f"""
    SELECT {", ".join(expressoes)}
    FROM read_parquet(?) AS o
    LEFT JOIN read_parquet(?) AS n
        ON o.CD_SETOR = n.CD_SETOR
    """

    linha = con.execute(
        consulta,
        [arquivo_dom1, arquivo_nacional]
    ).fetchone()

    for nome, divergencias in zip(bloco, linha):
        resultados_dom1.append({
            "conjunto": "DOM1",
            "variavel": nome,
            "divergencias": int(divergencias)
        })

    print(
        f"Bloco {inicio // tamanho_bloco + 1}: "
        f"{len(bloco)} variáveis verificadas"
    )

relatorio_dom1 = pd.DataFrame(resultados_dom1)

total_divergencias = int(
    relatorio_dom1["divergencias"].sum()
)

print("\nRESULTADO FINAL")
print("-" * 55)
print("Variáveis verificadas:", len(relatorio_dom1))
print("Total de divergências:", total_divergencias)

if total_divergencias == 0:
    print("DOM1: todas as variáveis verificadas sem divergências.")
else:
    print("ATENÇÃO: foram encontradas divergências.")
    display(
        relatorio_dom1[
            relatorio_dom1["divergencias"] > 0
        ]
    )


VALIDAÇÃO INTEGRAL — DOM1
-------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Bloco 1: 20 variáveis verificadas
Bloco 2: 20 variáveis verificadas
Bloco 3: 20 variáveis verificadas
Bloco 4: 20 variáveis verificadas
Bloco 5: 9 variáveis verificadas

RESULTADO FINAL
-------------------------------------------------------
Variáveis verificadas: 89
Total de divergências: 0
DOM1: todas as variáveis verificadas sem divergências.


*21.4. Validação integral dos demais conjuntos temáticos*

Comparar todas as variáveis dos conjuntos DOM2, DOM3, DEM, PAR, COR e ALF com o banco nacional integrado, utilizando CD_SETOR como chave. As comparações são realizadas em blocos de 20 variáveis, com registro do número de divergências por variável.

In [15]:

import os
import pandas as pd
import pyarrow.parquet as pq

# Incorporar os resultados já calculados do DOM1
resultados_validacao = resultados_dom1.copy()

# Validar automaticamente todos os demais conjuntos
conjuntos = ["DOM2", "DOM3", "DEM", "PAR", "COR", "ALF"]

resultados_validacao = resultados_dom1.copy()

for conjunto in conjuntos:

    arquivo_origem = os.path.join(
        pasta_tratados,
        f"{conjunto}_2022.parquet"
    )

    variaveis = [
        nome
        for nome in pq.ParquetFile(arquivo_origem).schema_arrow.names
        if nome != "CD_SETOR"
    ]

    print(f"\nVALIDAÇÃO — {conjunto}")
    print("-" * 50)

    resultados_conjunto = []

    for inicio in range(0, len(variaveis), 20):

        bloco = variaveis[inicio:inicio + 20]

        expressoes = [
            f"""
            COUNT(*) FILTER (
                WHERE o.{identificar(nome)}
                IS DISTINCT FROM n.{identificar(nome)}
            ) AS {identificar(nome)}
            """
            for nome in bloco
        ]

        consulta = f"""
        SELECT {", ".join(expressoes)}
        FROM read_parquet(?) AS o
        LEFT JOIN read_parquet(?) AS n
            ON o.CD_SETOR = n.CD_SETOR
        """

        linha = con.execute(
            consulta,
            [arquivo_origem, arquivo_nacional]
        ).fetchone()

        for nome, divergencias in zip(bloco, linha):
            resultados_conjunto.append({
                "conjunto": conjunto,
                "variavel": nome,
                "divergencias": int(divergencias)
            })

        print(
            f"Bloco {inicio // 20 + 1}: "
            f"{len(bloco)} variáveis verificadas"
        )

    df_conjunto = pd.DataFrame(resultados_conjunto)

 # Acumular os resultados da validação
resultados_validacao.extend(resultados_conjunto)


    total = int(df_conjunto["divergencias"].sum())

    print(f"Variáveis verificadas: {len(df_conjunto)}")
    print(f"Divergências: {total}")

    if total == 0:
        print(f"RESULTADO: {conjunto} sem divergências.")
    else:
        print(f"ATENÇÃO: divergências em {conjunto}")
        display(
            df_conjunto[df_conjunto["divergencias"] > 0]
        )

print("\nRESUMO DA VALIDAÇÃO TEMÁTICA")
print("-" * 50)

resumo = resultados_validacao.groupby(
    "conjunto", as_index=False
).agg(
    variaveis=("variavel", "count"),
    divergencias=("divergencias", "sum")
)

display(resumo)

print(
    "TOTAL DE VARIÁVEIS:",
    len(resultados_validacao)
)

print(
    "TOTAL DE DIVERGÊNCIAS:",
    int(resultados_validacao["divergencias"].sum())
)

relatorio_validacao = pd.DataFrame(resultados_validacao)



VALIDAÇÃO — DOM3
--------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Bloco 1: 20 variáveis verificadas


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Bloco 2: 20 variáveis verificadas


...

Bloco 15: 20 variáveis verificadas
Bloco 16: 20 variáveis verificadas
Bloco 17: 20 variáveis verificadas


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Bloco 18: 20 variáveis verificadas
Bloco 19: 2 variáveis verificadas
Variáveis verificadas: 362
Divergências: 0
RESULTADO: ALF sem divergências.

RESUMO DA VALIDAÇÃO TEMÁTICA
--------------------------------------------------


,conjunto,variaveis,divergencias
0,ALF,362,0
1,COR,95,0
2,DEM,36,0
3,DOM1,89,0
4,DOM2,406,0
5,DOM3,148,0
6,PAR,182,0


TOTAL DE VARIÁVEIS: 1318
TOTAL DE DIVERGÊNCIAS: 0


*21.6. Validação integral das variáveis do arquivo Básico*

Esta etapa compara as variáveis do arquivo Básico com os respectivos valores no banco nacional integrado, utilizando CD_SETOR como chave. A comparação abrange todos os setores censitários e distingue valores nulos de valores não nulos.

In [16]:

# VALIDAÇÃO INTEGRAL DO BÁSICO

arquivo_basico = os.path.join(
    pasta_tratados, "BASICO_2022.parquet"
)

# Identificar as variáveis, excluindo CD_SETOR
variaveis_basico = [
    nome
    for nome in pq.ParquetFile(arquivo_basico).schema_arrow.names
    if nome != "CD_SETOR"
]

resultados_basico = []

print("VALIDAÇÃO INTEGRAL — BÁSICO")
print("-" * 55)

# Comparar em blocos de 20 variáveis
for inicio in range(0, len(variaveis_basico), 20):

    bloco = variaveis_basico[inicio:inicio + 20]

    expressoes = [
        f"""
        COUNT(*) FILTER (
            WHERE o.{identificar(nome)}
            IS DISTINCT FROM n.{identificar(nome)}
        ) AS {identificar(nome)}
        """
        for nome in bloco
    ]

    consulta = f"""
    SELECT {", ".join(expressoes)}
    FROM read_parquet(?) AS o
    LEFT JOIN read_parquet(?) AS n
        ON o.CD_SETOR = n.CD_SETOR
    """

    linha = con.execute(
        consulta,
        [arquivo_basico, arquivo_nacional]
    ).fetchone()

    for nome, divergencias in zip(bloco, linha):
        resultados_basico.append({
            "conjunto": "BASICO",
            "variavel": nome,
            "divergencias": int(divergencias)
        })

    print(
        f"Bloco {inicio // 20 + 1}: "
        f"{len(bloco)} variáveis verificadas"
    )

relatorio_basico = pd.DataFrame(resultados_basico)

total_divergencias = int(
    relatorio_basico["divergencias"].sum()
)

print("\nRESULTADO FINAL — BÁSICO")
print("-" * 55)

print("Variáveis verificadas:", len(relatorio_basico))
print("Total de divergências:", total_divergencias)

if total_divergencias == 0:
    print("BÁSICO: todas as variáveis sem divergências.")
else:
    print("ATENÇÃO: divergências encontradas!")
    display(
        relatorio_basico[
            relatorio_basico["divergencias"] > 0
        ]
    )

# Incorporar ao relatório consolidado
relatorio_validacao = pd.concat(
    [relatorio_validacao, relatorio_basico],
    ignore_index=True
)

print("\nTOTAL DE VARIÁVEIS AUDITADAS:",
      len(relatorio_validacao))


VALIDAÇÃO INTEGRAL — BÁSICO
-------------------------------------------------------
Bloco 1: 20 variáveis verificadas


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Bloco 2: 17 variáveis verificadas

RESULTADO FINAL — BÁSICO
-------------------------------------------------------
Variáveis verificadas: 37
Total de divergências: 0
BÁSICO: todas as variáveis sem divergências.

TOTAL DE VARIÁVEIS AUDITADAS: 1355


*21.7. Verificação dos valores ausentes nos setores exclusivos do Básico*

Esta etapa identifica os setores presentes no arquivo Básico e ausentes nos conjuntos temáticos. Para esses registros, verifica se todas as 1.318 variáveis temáticas estão nulas no banco nacional. O procedimento permite identificar preenchimentos indevidos após a integração territorial.

In [17]:

# VERIFICAÇÃO DOS SETORES EXCLUSIVOS DO BÁSICO

# Identificar os setores que existem apenas no Básico
arquivo_dom1 = os.path.join(
    pasta_tratados, "DOM1_2022.parquet"
)

# Todas as colunas temáticas já identificadas na auditoria
colunas_tematicas = (
    relatorio_validacao.loc[
        relatorio_validacao["conjunto"] != "BASICO",
        "variavel"
    ].tolist()
)

assert len(colunas_tematicas) == 1318, (
    "Quantidade inesperada de variáveis temáticas."
)

# Comparação em blocos para limitar o uso de memória
tamanho_bloco = 20
divergencias_nulos = 0
setores_exclusivos = None
colunas_com_valores = []

print("VALIDAÇÃO DOS SETORES EXCLUSIVOS DO BÁSICO")
print("-" * 55)

for inicio in range(0, len(colunas_tematicas), tamanho_bloco):

    bloco = colunas_tematicas[inicio:inicio + tamanho_bloco]

    expressoes = [
        f"""
        COUNT(*) FILTER (
            WHERE n.{identificar(nome)} IS NOT NULL
        ) AS {identificar(nome)}
        """
        for nome in bloco
    ]

    consulta = f"""
    SELECT
        COUNT(*) AS total_setores,
        {", ".join(expressoes)}
    FROM read_parquet(?) AS n
    WHERE NOT EXISTS (
        SELECT 1
        FROM read_parquet(?) AS o
        WHERE o.CD_SETOR = n.CD_SETOR
    )
    """

    linha = con.execute(
        consulta,
        [arquivo_nacional, arquivo_dom1]
    ).fetchone()

    total_bloco = int(linha[0])

    if setores_exclusivos is None:
        setores_exclusivos = total_bloco
    else:
        assert setores_exclusivos == total_bloco, (
            "Contagem inconsistente de setores exclusivos."
        )

    for nome, quantidade in zip(bloco, linha[1:]):
        quantidade = int(quantidade)
        divergencias_nulos += quantidade

        if quantidade > 0:
            colunas_com_valores.append({
                "variavel": nome,
                "valores_nao_nulos": quantidade
            })

    print(
        f"Bloco {inicio // tamanho_bloco + 1}: "
        f"{len(bloco)} variáveis verificadas"
    )

print("\nRESULTADO FINAL")
print("-" * 55)
print("Setores exclusivos do Básico:", setores_exclusivos)
print("Variáveis temáticas verificadas:", len(colunas_tematicas))
print("Valores temáticos indevidamente preenchidos:", divergencias_nulos)

if setores_exclusivos != 9327:
    print("ATENÇÃO: quantidade inesperada de setores exclusivos.")

if divergencias_nulos == 0 and setores_exclusivos == 9327:
    print("RESULTADO: todos os valores temáticos esperados estão nulos.")
else:
    print("ATENÇÃO: foram identificadas inconsistências.")
    if colunas_com_valores:
        display(pd.DataFrame(colunas_com_valores))


VALIDAÇÃO DOS SETORES EXCLUSIVOS DO BÁSICO
-------------------------------------------------------
Bloco 1: 20 variáveis verificadas
Bloco 2: 20 variáveis verificadas
Bloco 3: 20 variáveis verificadas
Bloco 4: 20 variáveis verificadas
Bloco 5: 20 variáveis verificadas
Bloco 6: 20 variáveis verificadas
Bloco 7: 20 variáveis verificadas
Bloco 8: 20 variáveis verificadas
Bloco 9: 20 variáveis verificadas
Bloco 10: 20 variáveis verificadas
Bloco 11: 20 variáveis verificadas
Bloco 12: 20 variáveis verificadas
Bloco 13: 20 variáveis verificadas
Bloco 14: 20 variáveis verificadas
Bloco 15: 20 variáveis verificadas
Bloco 16: 20 variáveis verificadas
Bloco 17: 20 variáveis verificadas
Bloco 18: 20 variáveis verificadas
Bloco 19: 20 variáveis verificadas
Bloco 20: 20 variáveis verificadas
Bloco 21: 20 variáveis verificadas
Bloco 22: 20 variáveis verificadas
Bloco 23: 20 variáveis verificadas
Bloco 24: 20 variáveis verificadas
Bloco 25: 20 variáveis verificadas
Bloco 26: 20 variáveis verificadas


*21.8. Exportação do relatório de validação*

Esta etapa registra os resultados da auditoria de correspondência dos dados, incluindo as 1.355 variáveis verificadas e a conferência dos setores censitários exclusivos do arquivo Básico. O relatório será utilizado na documentação metodológica e na preparação dos dados para divulgação científica.

In [18]:

# EXPORTAÇÃO DO RELATÓRIO DE VALIDAÇÃO

import os
import pandas as pd

pasta_validacao = os.path.join(
    pasta_resultados, "VALIDACAO_2022"
)

os.makedirs(pasta_validacao, exist_ok=True)

# Garantir que todas as variáveis foram auditadas
assert len(relatorio_validacao) == 1355
assert relatorio_validacao["variavel"].nunique() == 1355
assert relatorio_validacao["divergencias"].sum() == 0

# Relatório detalhado das variáveis
arquivo_detalhado = os.path.join(
    pasta_validacao,
    "RELATORIO_VARIAVEIS_2022.csv"
)

relatorio_validacao.to_csv(
    arquivo_detalhado,
    index=False,
    encoding="utf-8-sig"
)

# Resumo por conjunto de dados
resumo_validacao = (
    relatorio_validacao
    .groupby("conjunto", as_index=False)
    .agg(
        variaveis=("variavel", "count"),
        divergencias=("divergencias", "sum")
    )
)

arquivo_resumo = os.path.join(
    pasta_validacao,
    "RESUMO_VALIDACAO_2022.csv"
)

resumo_validacao.to_csv(
    arquivo_resumo,
    index=False,
    encoding="utf-8-sig"
)

# Resultado dos setores exclusivos do Básico
resumo_setores = pd.DataFrame([{
    "setores_exclusivos_basico": setores_exclusivos,
    "variaveis_tematicas_verificadas": len(colunas_tematicas),
    "valores_tematicos_nao_nulos": divergencias_nulos
}])

assert setores_exclusivos == 9327
assert len(colunas_tematicas) == 1318
assert divergencias_nulos == 0

arquivo_setores = os.path.join(
    pasta_validacao,
    "VALIDACAO_SETORES_EXCLUSIVOS_2022.csv"
)

resumo_setores.to_csv(
    arquivo_setores,
    index=False,
    encoding="utf-8-sig"
)

print("RELATÓRIOS DE VALIDAÇÃO SALVOS")
print("-" * 55)
print(arquivo_detalhado)
print(arquivo_resumo)
print(arquivo_setores)

print("\nRESULTADO FINAL")
print("Variáveis auditadas:", len(relatorio_validacao))
print("Divergências:", int(relatorio_validacao["divergencias"].sum()))
print("Setores exclusivos:", setores_exclusivos)
print("Valores temáticos indevidos:", divergencias_nulos)


RELATÓRIOS DE VALIDAÇÃO SALVOS
-------------------------------------------------------
/content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/RESULTADOS_2022/VALIDACAO_2022/RELATORIO_VARIAVEIS_2022.csv
/content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/RESULTADOS_2022/VALIDACAO_2022/RESUMO_VALIDACAO_2022.csv
/content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/RESULTADOS_2022/VALIDACAO_2022/VALIDACAO_SETORES_EXCLUSIVOS_2022.csv

RESULTADO FINAL
Variáveis auditadas: 1355
Divergências: 0
Setores exclusivos: 9327
Valores temáticos indevidos: 0


*22.1. Download e inspeção do dicionário oficial do IBGE*

Esta etapa realiza o download do dicionário de variáveis dos Agregados por Setores Censitários do Censo Demográfico 2022, disponibilizado pelo IBGE. O arquivo é armazenado na pasta de dados originais e sua estrutura é inspecionada para subsidiar a documentação das 1.356 colunas do banco nacional integrado.

In [20]:

# DOWNLOAD DO DICIONÁRIO OFICIAL

import os
import requests
import zipfile
import pandas as pd

url_dicionario = (
    "https://ftp.ibge.gov.br/Censos/"
    "Censo_Demografico_2022/"
    "Agregados_por_Setores_Censitarios/"
    "dicionario_de_dados_agregados_por_setores_censitarios_20260520.xlsx"
)

pasta_documentacao = os.path.join(
    pasta_drive,
    "DADOS_ORIGINAIS_2022",
    "DOCUMENTACAO_IBGE"
)

os.makedirs(pasta_documentacao, exist_ok=True)

arquivo_dicionario = os.path.join(
    pasta_documentacao,
    "dicionario_de_dados_agregados_por_setores_censitarios_20260520.xlsx"
)

if not os.path.exists(arquivo_dicionario):
    resposta = requests.get(url_dicionario, timeout=120)
    resposta.raise_for_status()

    with open(arquivo_dicionario, "wb") as arquivo:
        arquivo.write(resposta.content)

    print("Dicionário baixado do IBGE.")
else:
    print("Dicionário já encontrado no Google Drive.")

# Verificar a integridade básica do arquivo XLSX
assert zipfile.is_zipfile(arquivo_dicionario), (
    "O arquivo baixado não é um XLSX válido."
)

# Inspecionar a estrutura do dicionário
excel = pd.ExcelFile(arquivo_dicionario)

print("\nDICIONÁRIO OFICIAL DO IBGE")
print("-" * 55)
print("Arquivo:", os.path.basename(arquivo_dicionario))
print("Tamanho (KB):", round(
    os.path.getsize(arquivo_dicionario) / 1024, 2
))
print("Planilhas encontradas:")

for nome in excel.sheet_names:
    print(" -", nome)


Dicionário baixado do IBGE.

DICIONÁRIO OFICIAL DO IBGE
-------------------------------------------------------
Arquivo: dicionario_de_dados_agregados_por_setores_censitarios_20260520.xlsx
Tamanho (KB): 115.59
Planilhas encontradas:
 - Dicionário Básico
 - Siglas Básico
 - Dicionário não PCT
 - Dicionário PCT - Indígenas
 - Dicionário PCT - Quilombolas


*22.2. Inspeção da estrutura dos dicionários oficiais*

Esta etapa examina as planilhas de documentação dos agregados básicos e temáticos do Censo Demográfico 2022, identificando seus campos, cabeçalhos e registros iniciais. A análise subsidia a construção do dicionário integrado e a associação das variáveis padronizadas às descrições oficiais do IBGE.

In [21]:

# INSPEÇÃO DOS DICIONÁRIOS

import pandas as pd

planilhas_analisar = [
    "Dicionário Básico",
    "Siglas Básico",
    "Dicionário não PCT"
]

for nome_planilha in planilhas_analisar:

    print("\n" + "=" * 65)
    print("PLANILHA:", nome_planilha)
    print("=" * 65)

    # Ler sem assumir que a primeira linha contém o cabeçalho
    df = pd.read_excel(
        arquivo_dicionario,
        sheet_name=nome_planilha,
        header=None,
        nrows=12,
        dtype=str
    )

    print("Colunas identificadas:", df.shape[1])
    print("Primeiras 12 linhas:")

    display(df.fillna("").iloc[:, :12])



PLANILHA: Dicionário Básico
Colunas identificadas: 3
Primeiras 12 linhas:


,0,1,2
0,Tema,Variável,Descrição
1,Básico,V0001,Total de pessoas
2,Básico,V0002,Total de Domicílios (DPPO + DPPV + DPPUO + DPI...
3,Básico,V0003,Total de Domicílios Particulares (DPPO + DPPV ...
4,Básico,V0004,Total de Domicílios Coletivos (DCCM + DCSM)
5,Básico,V0005,Média de moradores em Domicílios Particulares ...
6,Básico,V0006,Percentual de Domicílios Particulares Ocupados...
7,Básico,V0007,Total de Domicílios Particulares Ocupados (DPP...
8,Básico,V0008,Total de Domicílios Particulares de Uso Ocasi...
9,Básico,V0009,Total de Domicílios Particulares Permanentes V...



PLANILHA: Siglas Básico
Colunas identificadas: 2
Primeiras 12 linhas:


,0,1
0,Sigla,Descrição
1,DPPO,Domicílio Particular Permanente Ocupado
2,DPIO,Domicílio Particular Improvisado Ocupado
3,DPPV,Domicílio Particular Permanente Vago
4,DPPUO,Domicílio Particular Permanente de Uso Ocasional
5,DCCM,Domicílio Coletivo Com Morador
6,DCSM,Domicílio Coletivo Sem Morador
7,DPO,Domicílio Particular Ocupado (DPPO + DPIO)



PLANILHA: Dicionário não PCT
Colunas identificadas: 4
Primeiras 12 linhas:


,0,1,2,3
0,Tipo,Tema,Variável,Descrição
1,Domicílio,Características do Domicílio - Parte 1,V00001,Domicílios Particulares Permanentes Ocupados
2,Domicílio,Características do Domicílio - Parte 1,V00002,Domicílios Particulares Improvisados Ocupados
3,Domicílio,Características do Domicílio - Parte 1,V00003,Unidades de Habitação em Domicílios Coletivos ...
4,Domicílio,Características do Domicílio - Parte 1,V00004,Domicílios Particulares Permanentes Ocupados
5,Domicílio,Características do Domicílio - Parte 1,V00005,"Domicílios Particulares Permanentes Ocupados, ..."
6,Domicílio,Características do Domicílio - Parte 1,V00006,"Domicílios Particulares Improvisados Ocupados,..."
7,Domicílio,Características do Domicílio - Parte 1,V00007,"Domicílios Coletivos Com Morador, Quantidade d..."
8,Domicílio,Características do Domicílio - Parte 1,V00008,"Domicílios Particulares Permanentes Ocupados, ..."
9,Domicílio,Características do Domicílio - Parte 1,V00009,"Domicílios Particulares Improvisados Ocupados,..."


*22.3. Identificação dos temas e códigos das variáveis*

Esta etapa identifica os temas dos agregados censitários documentados pelo IBGE e verifica a quantidade de variáveis associadas a cada tema. O objetivo é estabelecer a correspondência entre as categorias oficiais e os prefixos utilizados na integração nacional, sem atribuir descrições incorretas às variáveis.

In [22]:

# IDENTIFICAÇÃO DOS TEMAS OFICIAIS

import pandas as pd

# Ler os dicionários completos
dic_basico = pd.read_excel(
    arquivo_dicionario,
    sheet_name="Dicionário Básico",
    dtype=str
)

dic_tematico = pd.read_excel(
    arquivo_dicionario,
    sheet_name="Dicionário não PCT",
    dtype=str
)

# Remover espaços dos nomes das colunas
dic_basico.columns = dic_basico.columns.str.strip()
dic_tematico.columns = dic_tematico.columns.str.strip()

print("DICIONÁRIO BÁSICO")
print("-" * 60)
print("Total de registros:", len(dic_basico))
print("Variáveis distintas:", dic_basico["Variável"].nunique())
print("Temas:")
print(dic_basico["Tema"].value_counts(dropna=False).to_string())

print("\nDICIONÁRIO TEMÁTICO NÃO PCT")
print("-" * 60)
print("Total de registros:", len(dic_tematico))

# Quantidade de variáveis por tipo e tema
resumo_temas = (
    dic_tematico
    .groupby(["Tipo", "Tema"], dropna=False)
    .agg(
        registros=("Variável", "size"),
        variaveis_distintas=("Variável", "nunique")
    )
    .reset_index()
)

display(resumo_temas)

# Verificar duplicações dentro de cada tema
duplicadas = dic_tematico.duplicated(
    subset=["Tema", "Variável"],
    keep=False
)

print("\nRegistros com tema e código repetidos:",
      duplicadas.sum())

if duplicadas.any():
    display(
        dic_tematico.loc[
            duplicadas,
            ["Tipo", "Tema", "Variável", "Descrição"]
        ].head(20)
    )


DICIONÁRIO BÁSICO
------------------------------------------------------------
Total de registros: 9
Variáveis distintas: 9
Temas:
Tema
Básico    9

DICIONÁRIO TEMÁTICO NÃO PCT
------------------------------------------------------------
Total de registros: 1411


,Tipo,Tema,registros,variaveis_distintas
0,Domicílio,Características do Domicílio - Parte 1,89,89
1,Domicílio,Características do Domicílio - Parte 2,406,406
2,Domicílio,Características do Domicílio - Parte 3,148,148
3,Mortalidade,Óbitos,93,93
4,Pessoas,Alfabetização,362,362
5,Pessoas,Cor ou Raça,95,95
6,Pessoas,Demografia,36,36
7,Pessoas,Parentesco,182,182



Registros com tema e código repetidos: 0


*22.4. Correspondência entre o arquivo Básico e o dicionário oficial*

Esta etapa verifica a correspondência entre os nomes das colunas do arquivo Básico e os códigos documentados pelo IBGE. Os campos sem descrição na planilha oficial são identificados separadamente para posterior documentação, sem atribuição automática de significados.

In [23]:

# CAMPOS DO BÁSICO E DOCUMENTAÇÃO OFICIAL

import os
import pandas as pd
import pyarrow.parquet as pq

arquivo_basico = os.path.join(
    pasta_tratados,
    "BASICO_2022.parquet"
)

# Ler somente o esquema, sem carregar os registros
colunas_basico = (
    pq.ParquetFile(arquivo_basico)
    .schema_arrow.names
)

# Preparar correspondência com o dicionário do IBGE
referencias_basico = dic_basico.copy()

referencias_basico["codigo_normalizado"] = (
    referencias_basico["Variável"]
    .astype(str)
    .str.strip()
    .str.upper()
)

mapa_descricoes = dict(zip(
    referencias_basico["codigo_normalizado"],
    referencias_basico["Descrição"]
))

# Construir tabela preliminar
campos_basico = pd.DataFrame({
    "coluna_banco": colunas_basico
})

campos_basico["codigo_normalizado"] = (
    campos_basico["coluna_banco"]
    .str.strip()
    .str.upper()
)

campos_basico["descricao_ibge"] = (
    campos_basico["codigo_normalizado"]
    .map(mapa_descricoes)
)

campos_basico["situacao_documentacao"] = (
    campos_basico["descricao_ibge"]
    .notna()
    .map({
        True: "Descrição localizada",
        False: "Descrição não localizada nesta planilha"
    })
)

print("DOCUMENTAÇÃO DO ARQUIVO BÁSICO")
print("-" * 60)
print("Colunas:", len(campos_basico))
print(
    "Com descrição oficial:",
    campos_basico["descricao_ibge"].notna().sum()
)
print(
    "Sem descrição nesta planilha:",
    campos_basico["descricao_ibge"].isna().sum()
)

display(campos_basico[
    ["coluna_banco", "descricao_ibge",
     "situacao_documentacao"]
])


DOCUMENTAÇÃO DO ARQUIVO BÁSICO
------------------------------------------------------------
Colunas: 38
Com descrição oficial: 9
Sem descrição nesta planilha: 29


,coluna_banco,descricao_ibge,situacao_documentacao
0,CD_SETOR,NaN,Descrição não localizada nesta planilha
1,SITUACAO,NaN,Descrição não localizada nesta planilha
2,CD_SIT,NaN,Descrição não localizada nesta planilha
3,CD_TIPO,NaN,Descrição não localizada nesta planilha
4,AREA_KM2,NaN,Descrição não localizada nesta planilha
5,CD_REGIAO,NaN,Descrição não localizada nesta planilha
6,NM_REGIAO,NaN,Descrição não localizada nesta planilha
7,CD_UF,NaN,Descrição não localizada nesta planilha
8,NM_UF,NaN,Descrição não localizada nesta planilha
9,CD_MUN,NaN,Descrição não localizada nesta planilha


*22.5. Construção do dicionário integrado de variáveis*

Esta etapa relaciona as 1.356 colunas do banco nacional às descrições disponíveis no dicionário oficial do IBGE. A correspondência considera o tema e o código de cada variável. Os campos territoriais sem descrição na planilha consultada permanecem identificados para documentação complementar. O resultado é exportado em CSV e Excel.

In [24]:

# CONSTRUÇÃO DO DICIONÁRIO INTEGRADO

import os
import pandas as pd
import pyarrow.parquet as pq

# 1. Correspondência entre prefixos e temas oficiais

mapa_temas = {
    "DOM1": "Características do Domicílio - Parte 1",
    "DOM2": "Características do Domicílio - Parte 2",
    "DOM3": "Características do Domicílio - Parte 3",
    "DEM": "Demografia",
    "PAR": "Parentesco",
    "COR": "Cor ou Raça",
    "ALF": "Alfabetização"
}

# 2. Preparar os dicionários oficiais

basico_ref = dic_basico.copy()
tematico_ref = dic_tematico.copy()

for df in [basico_ref, tematico_ref]:
    df["Variável"] = df["Variável"].astype(str).str.strip().str.upper()

tematico_ref["Tema"] = tematico_ref["Tema"].astype(str).str.strip()

# Verificar a unicidade das referências
assert not basico_ref["Variável"].duplicated().any()
assert not tematico_ref.duplicated(
    ["Tema", "Variável"]
).any()

mapa_basico = basico_ref.set_index(
    "Variável"
)["Descrição"].to_dict()

mapa_tematico = {
    (linha["Tema"], linha["Variável"]): linha["Descrição"]
    for _, linha in tematico_ref.iterrows()
}

# 3. Recuperar todas as colunas do banco nacional

colunas_nacional = (
    pq.ParquetFile(arquivo_nacional).schema_arrow.names
)

registros = []

for posicao, coluna in enumerate(colunas_nacional, start=1):

    if "_" in coluna and coluna.split("_", 1)[0] in mapa_temas:

        prefixo, codigo = coluna.split("_", 1)
        tema = mapa_temas[prefixo]

        descricao = mapa_tematico.get(
            (tema, codigo.upper())
        )

        conjunto = prefixo
        codigo_original = codigo

    else:
        conjunto = "BASICO"
        tema = "Básico"
        codigo_original = coluna

        descricao = mapa_basico.get(coluna.upper())

    if pd.notna(descricao) and str(descricao).strip():
        situacao = "Descrição oficial identificada"
    else:
        descricao = None
        situacao = "Documentação complementar necessária"

    registros.append({
        "ordem": posicao,
        "coluna_banco": coluna,
        "conjunto": conjunto,
        "tema_ibge": tema,
        "codigo_original": codigo_original,
        "descricao_ibge": descricao,
        "situacao_documentacao": situacao
    })

dicionario_integrado = pd.DataFrame(registros)

# 4. Verificar a integridade do dicionário

assert len(dicionario_integrado) == 1356
assert dicionario_integrado["coluna_banco"].is_unique
assert dicionario_integrado["coluna_banco"].tolist() == colunas_nacional

resumo_dicionario = (
    dicionario_integrado
    .groupby("conjunto", as_index=False)
    .agg(
        colunas=("coluna_banco", "count"),
        descricoes_identificadas=(
            "descricao_ibge",
            lambda x: x.notna().sum()
        )
    )
)

print("DICIONÁRIO INTEGRADO — CENSO 2022")
print("-" * 60)

display(resumo_dicionario)

print("Total de colunas:", len(dicionario_integrado))
print(
    "Descrições identificadas:",
    dicionario_integrado["descricao_ibge"].notna().sum()
)
print(
    "Descrições pendentes:",
    dicionario_integrado["descricao_ibge"].isna().sum()
)

# Verificar se as únicas pendências são os 29 campos territoriais
pendentes = dicionario_integrado[
    dicionario_integrado["descricao_ibge"].isna()
]

assert len(pendentes) == 29, (
    "Quantidade inesperada de descrições pendentes. "
    "Verifique a correspondência entre os códigos."
)

assert (pendentes["conjunto"] == "BASICO").all(), (
    "Há variáveis temáticas sem descrição."
)

# 5. Exportar a documentação

pasta_dicionario = os.path.join(
    pasta_resultados,
    "DICIONARIO_2022"
)

os.makedirs(pasta_dicionario, exist_ok=True)

arquivo_csv = os.path.join(
    pasta_dicionario,
    "DICIONARIO_INTEGRADO_2022.csv"
)

arquivo_excel = os.path.join(
    pasta_dicionario,
    "DICIONARIO_INTEGRADO_2022.xlsx"
)

dicionario_integrado.to_csv(
    arquivo_csv,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(arquivo_excel, engine="openpyxl") as writer:

    dicionario_integrado.to_excel(
        writer,
        sheet_name="Dicionario integrado",
        index=False
    )

    resumo_dicionario.to_excel(
        writer,
        sheet_name="Resumo",
        index=False
    )

    pendentes.to_excel(
        writer,
        sheet_name="Documentacao pendente",
        index=False
    )

print("\nARQUIVOS SALVOS")
print(arquivo_csv)
print(arquivo_excel)


DICIONÁRIO INTEGRADO — CENSO 2022
------------------------------------------------------------


,conjunto,colunas,descricoes_identificadas
0,ALF,362,362
1,BASICO,38,9
2,COR,95,95
3,DEM,36,36
4,DOM1,89,89
5,DOM2,406,406
6,DOM3,148,148
7,PAR,182,182


Total de colunas: 1356
Descrições identificadas: 1327
Descrições pendentes: 29

ARQUIVOS SALVOS
/content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/RESULTADOS_2022/DICIONARIO_2022/DICIONARIO_INTEGRADO_2022.csv
/content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/RESULTADOS_2022/DICIONARIO_2022/DICIONARIO_INTEGRADO_2022.xlsx


*22.6. Complementação do dicionário dos campos territoriais*

Esta etapa incorpora as descrições dos 29 campos territoriais do arquivo Básico, com base na Nota metodológica nº 06 do IBGE. O procedimento registra a fonte documental e atualiza o dicionário integrado, sem alterar os dados censitários.

In [25]:

# COMPLEMENTAR O DICIONÁRIO TERRITORIAL

import os
import pandas as pd

fonte_territorial = (
    "IBGE. Censo Demográfico 2022. "
    "Agregados por Setores Censitários. "
    "Nota metodológica n. 06."
)

url_fonte = (
    "https://biblioteca.ibge.gov.br/"
    "visualizacao/livros/liv102136.pdf"
)

# Descrições dos campos territoriais do arquivo Básico
descricoes_territoriais = {
    "CD_SETOR": "Geocódigo do Setor Censitário",
    "SITUACAO": "Situação do Setor Censitário",
    "CD_SIT": "Situação detalhada do Setor Censitário",
    "CD_TIPO": "Tipo do Setor Censitário",
    "AREA_KM2": "Área do Setor Censitário em quilômetros quadrados",
    "CD_REGIAO": "Código da Grande Região",
    "NM_REGIAO": "Nome da Grande Região",
    "CD_UF": "Código da Unidade da Federação",
    "NM_UF": "Nome da Unidade da Federação",
    "CD_MUN": "Código do Município",
    "NM_MUN": "Nome do Município",
    "CD_DIST": "Código do Distrito",
    "NM_DIST": "Nome do Distrito",
    "CD_SUBDIST": "Código do Subdistrito",
    "NM_SUBDIST": "Nome do Subdistrito",
    "CD_BAIRRO": "Código do Bairro",
    "NM_BAIRRO": "Nome do Bairro",
    "CD_NU": "Código do Núcleo Urbano",
    "NM_NU": "Nome do Núcleo Urbano",
    "CD_FCU": "Código da Favela e Comunidade Urbana",
    "NM_FCU": "Nome da Favela e Comunidade Urbana",
    "CD_AGLOM": "Código do Aglomerado",
    "NM_AGLOM": "Nome do Aglomerado",
    "CD_RGINT": "Código da Região Geográfica Intermediária",
    "NM_RGINT": "Nome da Região Geográfica Intermediária",
    "CD_RGI": "Código da Região Geográfica Imediata",
    "NM_RGI": "Nome da Região Geográfica Imediata",
    "CD_CONCURB": "Código da Concentração Urbana",
    "NM_CONCURB": "Nome da Concentração Urbana"
}

# Verificar se a lista corresponde exatamente às pendências
pendentes_atuais = set(
    dicionario_integrado.loc[
        dicionario_integrado["descricao_ibge"].isna(),
        "coluna_banco"
    ]
)

assert pendentes_atuais == set(descricoes_territoriais), (
    "Os campos pendentes não correspondem à documentação."
)

# Trabalhar sobre uma cópia do dicionário
dicionario_completo = dicionario_integrado.copy()

# Registrar a fonte de cada descrição
dicionario_completo["fonte_descricao"] = (
    "IBGE — Dicionário de dados, 20/05/2026"
)
dicionario_completo["url_fonte_complementar"] = None

# Preencher somente as 29 descrições ausentes
mascara = dicionario_completo["coluna_banco"].isin(
    descricoes_territoriais
)

dicionario_completo.loc[mascara, "descricao_ibge"] = (
    dicionario_completo.loc[mascara, "coluna_banco"]
    .map(descricoes_territoriais)
)

dicionario_completo.loc[
    mascara, "situacao_documentacao"
] = "Descrição oficial identificada"

dicionario_completo.loc[
    mascara, "fonte_descricao"
] = fonte_territorial

dicionario_completo.loc[
    mascara, "url_fonte_complementar"
] = url_fonte

# Validar a documentação
assert len(dicionario_completo) == 1356
assert dicionario_completo["coluna_banco"].is_unique
assert dicionario_completo["descricao_ibge"].notna().all()

# Salvar sem sobrescrever o dicionário preliminar
arquivo_csv_completo = os.path.join(
    pasta_dicionario,
    "DICIONARIO_COMPLETO_2022.csv"
)

arquivo_excel_completo = os.path.join(
    pasta_dicionario,
    "DICIONARIO_COMPLETO_2022.xlsx"
)

dicionario_completo.to_csv(
    arquivo_csv_completo,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    arquivo_excel_completo,
    engine="openpyxl"
) as writer:

    dicionario_completo.to_excel(
        writer,
        sheet_name="Dicionario completo",
        index=False
    )

    dicionario_completo.loc[mascara].to_excel(
        writer,
        sheet_name="Campos territoriais",
        index=False
    )

print("DICIONÁRIO COMPLETO — CENSO 2022")
print("-" * 55)
print("Colunas documentadas:", len(dicionario_completo))
print(
    "Descrições pendentes:",
    dicionario_completo["descricao_ibge"].isna().sum()
)
print("Campos territoriais complementados:", mascara.sum())

print("\nARQUIVOS SALVOS")
print(arquivo_csv_completo)
print(arquivo_excel_completo)


DICIONÁRIO COMPLETO — CENSO 2022
-------------------------------------------------------
Colunas documentadas: 1356
Descrições pendentes: 0
Campos territoriais complementados: 29

ARQUIVOS SALVOS
/content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/RESULTADOS_2022/DICIONARIO_2022/DICIONARIO_COMPLETO_2022.csv
/content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/RESULTADOS_2022/DICIONARIO_2022/DICIONARIO_COMPLETO_2022.xlsx


*23.1. Organização dos arquivos para divulgação científica*

Esta etapa prepara uma pasta com o dicionário integrado e os relatórios de validação do banco nacional do Censo Demográfico 2022. Os arquivos são copiados sem modificar os originais, para posterior incorporação ao repositório GitHub e ao depósito científico no Zenodo.

In [26]:

# PREPARAÇÃO DA DOCUMENTAÇÃO

import os
import shutil

pasta_publicacao = os.path.join(
    pasta_resultados,
    "PUBLICACAO_2022"
)

os.makedirs(pasta_publicacao, exist_ok=True)

arquivos_publicacao = [
    os.path.join(
        pasta_dicionario,
        "DICIONARIO_COMPLETO_2022.csv"
    ),
    os.path.join(
        pasta_dicionario,
        "DICIONARIO_COMPLETO_2022.xlsx"
    ),
    os.path.join(
        pasta_resultados,
        "VALIDACAO_2022",
        "RELATORIO_VARIAVEIS_2022.csv"
    ),
    os.path.join(
        pasta_resultados,
        "VALIDACAO_2022",
        "RESUMO_VALIDACAO_2022.csv"
    ),
    os.path.join(
        pasta_resultados,
        "VALIDACAO_2022",
        "VALIDACAO_SETORES_EXCLUSIVOS_2022.csv"
    )
]

print("PREPARAÇÃO DA DOCUMENTAÇÃO")
print("-" * 60)

for origem in arquivos_publicacao:

    if not os.path.isfile(origem):
        raise FileNotFoundError(
            f"Arquivo não encontrado: {origem}"
        )

    destino = os.path.join(
        pasta_publicacao,
        os.path.basename(origem)
    )

    shutil.copy2(origem, destino)

    print("Copiado:", os.path.basename(destino))

print("\nDOCUMENTAÇÃO ORGANIZADA")
print("Pasta:", pasta_publicacao)
print("Total de arquivos:", len(arquivos_publicacao))


PREPARAÇÃO DA DOCUMENTAÇÃO
------------------------------------------------------------
Copiado: DICIONARIO_COMPLETO_2022.csv
Copiado: DICIONARIO_COMPLETO_2022.xlsx
Copiado: RELATORIO_VARIAVEIS_2022.csv
Copiado: RESUMO_VALIDACAO_2022.csv
Copiado: VALIDACAO_SETORES_EXCLUSIVOS_2022.csv

DOCUMENTAÇÃO ORGANIZADA
Pasta: /content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/RESULTADOS_2022/PUBLICACAO_2022
Total de arquivos: 5


*23.2. Elaboração da documentação geral do banco nacional*

Esta etapa gera o arquivo README.md, com informações sobre as fontes dos dados, abrangência territorial, estrutura das variáveis, metodologia de integração, validação e limitações. O documento acompanhará o notebook no GitHub e o conjunto de dados no Zenodo.

In [27]:

#  GERAR README DO BANCO NACIONAL

import os
from textwrap import dedent

readme = dedent("""\
# Brazilian Population Census 2022
## Integrated National Dataset of Census Tract Aggregates

### Overview

This dataset integrates publicly available census tract-level
aggregates from the Brazilian Population Census 2022, produced
by the Instituto Brasileiro de Geografia e Estatística (IBGE).

The integrated database was prepared to support reproducible
spatial analysis of demographic, socioeconomic, and
housing-related characteristics across Brazil.

This is a research-derived integration of official IBGE data,
not an official dataset published by IBGE.

### Dataset structure

- Geographic coverage: Brazil
- Reference year: 2022
- Spatial unit: Census tract (setor censitário)
- Number of records: 468,099
- Number of columns: 1,356
- Geographic identifier: CD_SETOR
- Data format: Apache Parquet

The database includes 29 identification and territorial
characterization fields, 9 basic statistical indicators,
and 1,318 thematic variables.

### Integrated sources

| Code | Dataset | Variables |
|---|---|---:|
| BASICO | Basic and territorial information | 38 columns |
| DOM1 | Household Characteristics - Part 1 | 89 |
| DOM2 | Household Characteristics - Part 2 | 406 |
| DOM3 | Household Characteristics - Part 3 | 148 |
| DEM | Demography | 36 |
| PAR | Household Relationships | 182 |
| COR | Color or Race | 95 |
| ALF | Literacy | 362 |

The listed variable counts for thematic datasets exclude
the common census tract identifier.

Mortality, Indigenous-specific, and Quilombola-specific
thematic datasets are not included in this release.

### Data processing

The processing workflow comprises:

1. Acquisition of official IBGE CSV archives.
2. Standardization of census tract identifiers.
3. Conversion of source datasets to Apache Parquet.
4. Preservation of thematic variable prefixes.
5. Integration using CD_SETOR as the unique key.
6. Preservation of all census tracts in the Basic dataset.
7. Consolidation of integration batches into a national file.
8. Validation against the processed source datasets.

The national database was integrated using a left join
strategy with the Basic dataset as the territorial reference.

### Data coverage and missing values

The Basic dataset contains 468,099 census tracts.

The thematic datasets contain 458,772 census tracts.
The remaining 9,327 census tracts are retained in the
integrated database, with null values in all thematic fields.

Original values were preserved as text during processing.
The statistical disclosure control marker X must not be
interpreted as zero or as a missing join value.

Numerical conversion, including treatment of decimal commas,
must be performed according to the requirements of each analysis.

### Validation

The integrated database was compared against the eight
processed Parquet source datasets.

Validation results:

- 1,355 data variables compared.
- Zero value discrepancies identified.
- 468,099 unique census tract records.
- 9,327 Basic-only census tracts verified.
- Zero non-null thematic values in those Basic-only tracts.

These checks establish consistency between the integrated
database and the processed source files. They do not
independently certify the accuracy of the original IBGE
statistics or all upstream processing decisions.

### Documentation files

- DICIONARIO_COMPLETO_2022.csv
- DICIONARIO_COMPLETO_2022.xlsx
- RELATORIO_VARIAVEIS_2022.csv
- RESUMO_VALIDACAO_2022.csv
- VALIDACAO_SETORES_EXCLUSIVOS_2022.csv

The integrated data dictionary includes variable identifiers,
source groups, thematic classifications, and descriptions.

The territorial field descriptions should be checked against
the corresponding IBGE methodological documentation before
final public release.

### Official data source

Instituto Brasileiro de Geografia e Estatística (IBGE).
Censo Demográfico 2022 - Agregados por Setores Censitários.

https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/

### Reproducible workflow

The computational notebook is available at:

https://github.com/PriscilaForcel/DOUTORADO

The workflow was developed using Python and Google Colab,
with supporting libraries including Pandas, PyArrow,
and DuckDB.

### Research context

This database was prepared within doctoral research in
Urban Engineering at the Federal University of São Carlos
(UFSCar), Brazil.

Research funding: Fundação de Amparo à Pesquisa do Estado
de São Paulo (FAPESP), grant 2025/08490-4.

Related research context: Centro de Estudos da Favela
(CEFAVELA), FAPESP grant 2022/12259-8.

### Citation and license

The dataset DOI, formal citation, version, and distribution
license will be added following final repository preparation
and verification of source-data reuse conditions.
""")

arquivo_readme = os.path.join(
    pasta_publicacao,
    "README.md"
)

with open(arquivo_readme, "w", encoding="utf-8") as f:
    f.write(readme)

print("README GERADO COM SUCESSO")
print("-" * 55)
print("Arquivo:", arquivo_readme)
print("Tamanho:", os.path.getsize(arquivo_readme), "bytes")
print("Linhas:", len(readme.splitlines()))


README GERADO COM SUCESSO
-------------------------------------------------------
Arquivo: /content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/RESULTADOS_2022/PUBLICACAO_2022/README.md
Tamanho: 4748 bytes
Linhas: 147


*23.3. Conferência dos arquivos de documentação*

Esta etapa verifica a presença dos arquivos de documentação preparados para divulgação científica e calcula suas assinaturas SHA-256. O procedimento permite identificar os arquivos de forma inequívoca e registrar sua integridade, sem alterar seus conteúdos.

In [28]:

# CONFERÊNCIA DA DOCUMENTAÇÃO

import os
import hashlib
import pandas as pd

arquivos_esperados = [
    "README.md",
    "DICIONARIO_COMPLETO_2022.csv",
    "DICIONARIO_COMPLETO_2022.xlsx",
    "RELATORIO_VARIAVEIS_2022.csv",
    "RESUMO_VALIDACAO_2022.csv",
    "VALIDACAO_SETORES_EXCLUSIVOS_2022.csv"
]

def calcular_sha256(caminho):
    hash_obj = hashlib.sha256()

    with open(caminho, "rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            hash_obj.update(bloco)

    return hash_obj.hexdigest()

registros = []

for nome in arquivos_esperados:
    caminho = os.path.join(pasta_publicacao, nome)

    if not os.path.isfile(caminho):
        raise FileNotFoundError(f"Arquivo ausente: {nome}")

    registros.append({
        "arquivo": nome,
        "tamanho_kb": round(os.path.getsize(caminho) / 1024, 2),
        "sha256": calcular_sha256(caminho)
    })

manifesto = pd.DataFrame(registros)

print("CONFERÊNCIA DA DOCUMENTAÇÃO")
print("-" * 60)
print("Arquivos encontrados:", len(manifesto))
print("Total esperado:", len(arquivos_esperados))

display(manifesto)

assert len(manifesto) == 6

print("\nRESULTADO: documentação localizada e verificada.")


CONFERÊNCIA DA DOCUMENTAÇÃO
------------------------------------------------------------
Arquivos encontrados: 6
Total esperado: 6


,arquivo,tamanho_kb,sha256
0,README.md,4.64,856959bb696624a807379a37eb3db4f7e4621d03696d7d...
1,DICIONARIO_COMPLETO_2022.csv,313.52,f1153f11f58cf9b7bc436405646dd24fecda56315bc908...
2,DICIONARIO_COMPLETO_2022.xlsx,70.17,b65fe66084c05a92fade92c877c88eceb3901a806bbcc9...
3,RELATORIO_VARIAVEIS_2022.csv,23.78,4fe8a728a627efe4486a40cff50addee0d5d672983e2df...
4,RESUMO_VALIDACAO_2022.csv,0.11,d1c41b4e208b95484b2dd28a2358d96d5aa399df622c27...
5,VALIDACAO_SETORES_EXCLUSIVOS_2022.csv,0.10,9405031f3b143d333fec2eaff7fd082dd0f2270f8a1854...



RESULTADO: documentação localizada e verificada.
